# Tiếp tục kiểm chứng nguồn không gian — chỉ HyperGAMO

Notebook này chỉ chạy suite `KLTN_GAN_Exposure_25d35efe55cc`, release
`4c08e7e92312`. Không chứa các cell baseline B. Các module khoa học được giữ nguyên.

1. Sau khi hai worker của phiên đang chạy đã dừng, ở notebook cũ chọn
   **Save Version → Quick Save → bật Save Output**. Chờ version lưu hoàn tất.
2. Import notebook này vào Kaggle. Gắn **Output đầy đủ mới nhất** của phiên vừa lưu
   vào Input, cùng nguồn `indian_pines_corrected.mat` và Standard GT đúng hash.
   ZIP `_REPORT.zip` không dùng để tiếp tục train.
3. Giữ cùng môi trường/package và GPU **T4 ×2** như phiên đã tạo checkpoint.
   Không tự nâng cấp môi trường; code sẽ chặn checkpoint khác contract.
4. Giữ `ACTION = "train"`, `JOB_FILTER = None`. Chọn **Save Version → Save & Run All**
   để chạy nền. Notebook này chỉ có một cell code, nên không chạy baseline B.
5. Xác nhận log in **“Đã có checkpoint”** và **“Khôi phục Output đã lưu”**.
   Các job đã xong được bỏ qua; job dở tiếp tục từ state đã commit.

`SESSION_HOURS = 8` là giới hạn thời gian mỗi worker, không phải ngân sách mới của
mô hình. Mỗi job vẫn kết thúc đúng 8.000 C updates. Chưa chấm test trong lần chạy này.
Nếu chỉ còn quota GPU thấp hơn, đặt giới hạn phiên phù hợp trước khi chạy.

Sau lần chạy nền, lần tiếp theo phải gắn **Output mới nhất của lần chạy nền đó**;
code không tự truy cập version mới nhất trên website. Có nhiều snapshot khác tiến độ
thì điền `RESUME_RUN_PATH` bằng thư mục suite muốn tiếp tục trong `/kaggle/input`.

Phạm vi thí nghiệm: can thiệp nguồn context trên toàn hệ thống HyperGAMO;
chưa tách riêng generator hoặc định lượng số điểm tăng của kết quả B cũ.


In [ ]:
# CHẠY RIÊNG CELL NÀY TRÊN KAGGLE; bật GPU trước.
# Chỉ tiếp tục checkpoint kiểm chứng đã lưu; không chạy các baseline B.
# Input cần Indian Pines corrected HSI + GT như notebook B, không chỉ ZIP báo cáo.
ACTION = "train"       # benchmark -> train -> test. prepare chỉ kiểm CPU.
BENCHMARK_C_UPDATES = 400   # 400 đầu tiên tính trong ngân sách 8000; tiếp tục từ checkpoint.
MAX_WORKERS = 2            # T4 x2: hai process độc lập; P100: tự dùng một.
GPU_IDS = None             # None tự chọn; ví dụ [0] để chỉ dùng GPU 0.
SESSION_HOURS = 8          # Hết thời gian: checkpoint, lần sau tiếp tục.
SAVE_EVERY = 100           # C updates; checkpoint theo cặp cập nhật.
HSI_PATH = None            # Tự tìm file .mat đúng hash trong /kaggle/input.
GT_PATH = None
RESUME_RUN_PATH = None     # Tự khôi phục một Output suite đã lưu; nhiều snapshot thì điền rõ.
OUTPUT_ROOT = "/kaggle/working"
JOB_FILTER = None          # train mặc định tất cả30; tùy chọn [(1,42,"blocked"), ...].


# Chỉ tiếp tục suite đã lưu; thiếu checkpoint thì dừng trước khi tạo runner/train.
from pathlib import Path as _ResumePath
import json as _resume_json

def _resume_require_existing(output_root, explicit=None, input_root="/kaggle/input"):
    suite_name = "KLTN_GAN_Exposure_25d35efe55cc"
    working = _ResumePath(output_root) / suite_name
    if working.is_dir():
        candidates = [working]
    elif explicit:
        candidates = [_ResumePath(explicit)]
    else:
        inputs = _ResumePath(input_root)
        candidates = sorted({p.parent for p in inputs.rglob("suite_manifest.json")
                             if p.parent.name == suite_name}) if inputs.is_dir() else []
    if not candidates:
        raise RuntimeError(
            "CHƯA CÓ CHECKPOINT ĐỂ TIẾP TỤC. Gắn toàn bộ Output mới nhất của "
            "suite KLTN_GAN_Exposure_25d35efe55cc vào Input, cùng hai MAT gốc. "
            "ZIP _REPORT.zip không có weights. Cell dừng, không train từ đầu."
        )
    for root in candidates:
        manifest = root / "suite_manifest.json"
        if (not manifest.is_file()
                or _resume_json.loads(manifest.read_text(encoding="utf-8")).get("suite_id")
                != "25d35efe55cc"):
            raise RuntimeError(f"Output không thuộc đúng suite đã khóa: {root}")
        required = [root / "data" / "data_manifest.json"]
        for region in (1, 2, 3):
            folder = root / "data" / f"region_{region}"
            required.extend(folder / name for name in (
                "geometry.npz", "preprocessing.npz", "region_manifest.json",
                "X_train_blocked.npy", "X_train_reuse_control.npy"))
        if any(not p.is_file() for p in required):
            raise RuntimeError(f"Output thiếu data/preprocessing để resume: {root}")
        states = sorted((root / "jobs").glob("*/recovery/state.json"))
        saved_updates = 0
        for state_path in states:
            state = _resume_json.loads(state_path.read_text(encoding="utf-8"))
            job = state_path.parent.parent
            pieces = state.get("checkpoint", {}).get("files", {})
            names = [_ResumePath(name).name for name in pieces]
            if (not pieces or not any(n.endswith(".index") for n in names)
                    or not any(".data-" in n for n in names)):
                raise RuntimeError(f"Checkpoint thiếu index/data: {state_path}")
            for relative in pieces:
                piece = (job / relative).resolve()
                if not piece.is_relative_to(job.resolve()) or not piece.is_file():
                    raise RuntimeError(f"Thiếu file checkpoint: {state_path} / {relative}")
            saved_updates += int(state.get("updates", 0))
        if saved_updates <= 0:
            raise RuntimeError(f"Chưa có bước train đã commit trong Output: {root}")
        print(f"Đã có checkpoint: {root}; {saved_updates:,}/240,000 C updates đã lưu.",
              flush=True)
    return candidates

_resume_require_existing(OUTPUT_ROOT, RESUME_RUN_PATH)

EXPOSURE_COMPONENTS = {
    'gan_exposure_data.py': '"""Verified train-only observations for the registered HyperGAMO intervention.\n\nNo work is performed on import. This module can be embedded in a Kaggle cell.\nAll context labels are excluded from training input construction. Test patches\nare constructed exclusively by the explicit exposure_load_test entry point.\n"""\nfrom pathlib import Path\nimport hashlib\nimport json\nimport os\nimport uuid\nimport numpy as np\n\n\nEXPOSURE_RAW_SHA256 = {\n    "hsi": "ec2f8808710919d566f70f0d4aa885aae1ddfd42b734aba71c5e12ca65450939",\n    "gt": "65c4687a8ab04f6da4789799bc3bc4f6e88bccac3ed6a2e6ae367e5e6b9e429c",\n}\nEXPOSURE_SELECTION_SHA256 = "c91ea95a6d91df3751b3b13643fb4545234b5e7565960e6b255cef4a887c912c"\nEXPOSURE_REGIONS = {\n    1: {"box": [15, 39, 15, 39], "train_n": 9373, "test_n": 437,\n        "affected_n": 285, "control_shared_raw_n": 596,\n        "train_counts": [46,1047,685,155,473,724,28,478,20,936,2455,476,205,1265,297,83]},\n    2: {"box": [15, 39, 105, 129], "train_n": 9554, "test_n": 330,\n        "affected_n": 260, "control_shared_raw_n": 515,\n        "train_counts": [46,1404,830,237,483,730,28,332,20,960,2252,593,205,955,386,93]},\n    3: {"box": [105, 129, 45, 69], "train_n": 9582, "test_n": 342,\n        "affected_n": 206, "control_shared_raw_n": 396,\n        "train_counts": [46,1428,767,237,395,495,28,478,20,972,2230,593,149,1265,386,93]},\n}\nEXPOSURE_CLASS_NAMES = ["Alfalfa", "Corn-notill", "Corn-mintill", "Corn",\n    "Grass-pasture", "Grass-trees", "Grass-pasture-mowed", "Hay-windrowed",\n    "Oats", "Soybean-notill", "Soybean-mintill", "Soybean-clean", "Wheat",\n    "Woods", "Buildings-Grass-Trees-Drives", "Stone-Steel-Towers"]\n\n\ndef exposure_sha_file(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for block in iter(lambda: stream.read(4 * 1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef exposure_sha_array(value):\n    array = np.ascontiguousarray(value)\n    digest = hashlib.sha256()\n    digest.update(str(array.dtype).encode("ascii"))\n    digest.update(json.dumps(list(array.shape), separators=(",", ":")).encode("ascii"))\n    digest.update(array.tobytes())\n    return digest.hexdigest()\n\n\ndef _exposure_json_bytes(value):\n    return json.dumps(value, sort_keys=True, ensure_ascii=True,\n                      separators=(",", ":"), allow_nan=False).encode("utf-8")\n\n\ndef _exposure_atomic_bytes(path, data):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")\n    try:\n        with temp.open("wb") as stream:\n            stream.write(data)\n            stream.flush()\n            os.fsync(stream.fileno())\n        os.replace(temp, path)\n    finally:\n        if temp.exists():\n            temp.unlink()\n\n\ndef _exposure_atomic_npz(path, **arrays):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")\n    try:\n        with temp.open("wb") as stream:\n            np.savez_compressed(stream, **arrays)\n            stream.flush()\n            os.fsync(stream.fileno())\n        os.replace(temp, path)\n    finally:\n        if temp.exists():\n            temp.unlink()\n\n\ndef exposure_discover_raw(explicit_hsi=None, explicit_gt=None,\n                          search_roots=("/kaggle/input",)):\n    """Resolve the exact verified MAT bytes, including harmless duplicate copies."""\n    chosen = {}\n    for kind, explicit in (("hsi", explicit_hsi), ("gt", explicit_gt)):\n        if explicit is not None:\n            path = Path(explicit).expanduser().resolve()\n            if not path.is_file() or exposure_sha_file(path) != EXPOSURE_RAW_SHA256[kind]:\n                raise ValueError("Wrong verified Indian Pines %s MAT: %s" % (kind, path))\n            chosen[kind] = path\n    needed = set(EXPOSURE_RAW_SHA256) - set(chosen)\n    if needed:\n        candidates = set()\n        for root in search_roots:\n            root = Path(root).expanduser()\n            if root.is_file() and root.suffix.lower() == ".mat":\n                candidates.add(root.resolve())\n            elif root.is_dir():\n                candidates.update(p.resolve() for p in root.rglob("*")\n                                  if p.is_file() and p.suffix.lower() == ".mat"\n                                  and "pines" in p.name.lower())\n        for path in sorted(candidates, key=lambda p: (len(str(p)), str(p))):\n            digest = exposure_sha_file(path)\n            for kind in sorted(needed):\n                if digest == EXPOSURE_RAW_SHA256[kind]:\n                    chosen[kind] = path\n            needed -= set(chosen)\n            if not needed:\n                break\n    if needed:\n        raise FileNotFoundError("Exact verified MAT missing: %s. Attach original "\n                                "indian_pines_corrected.mat and indian_pines_corrected_gt.mat; "\n                                "no substitute labels or transformed cube." % ", ".join(sorted(needed)))\n    return {"hsi_path": str(chosen["hsi"]), "gt_path": str(chosen["gt"]),\n            "hsi_sha256": EXPOSURE_RAW_SHA256["hsi"],\n            "gt_sha256": EXPOSURE_RAW_SHA256["gt"]}\n\n\ndef _exposure_load_raw(raw_paths):\n    from scipy.io import loadmat\n    arrays = {}\n    for kind, shape in (("hsi", (145,145,200)), ("gt", (145,145))):\n        path = Path(raw_paths[kind + "_path"])\n        if exposure_sha_file(path) != EXPOSURE_RAW_SHA256[kind]:\n            raise ValueError("Raw MAT hash changed: " + str(path))\n        document = loadmat(path)\n        candidates = [a for key, a in document.items() if not key.startswith("__")\n                      and isinstance(a, np.ndarray) and a.shape == shape\n                      and np.issubdtype(a.dtype, np.number)]\n        if len(candidates) != 1 or not np.isfinite(candidates[0]).all():\n            raise ValueError("Ambiguous/nonfinite MAT input " + kind)\n        arrays[kind] = candidates[0]\n    cube = arrays["hsi"].astype(np.float64)\n    gt = arrays["gt"]\n    if np.any(gt != np.floor(gt)) or set(np.unique(gt)) != set(range(17)):\n        raise ValueError("GT must contain exactly integer classes0..16")\n    gt = gt.astype(np.int16)\n    if np.count_nonzero(gt) != 10249:\n        raise ValueError("GT must contain exactly10249 labeled centers")\n    return cube, gt\n\n\ndef _exposure_sources(shape, coords):\n    offsets = np.arange(-5, 6, dtype=np.int32)\n    rr = coords[:, 0, None, None] + offsets[None, :, None]\n    cc = coords[:, 1, None, None] + offsets[None, None, :]\n    valid = (rr >= 0) & (rr < shape[0]) & (cc >= 0) & (cc < shape[1])\n    ids = np.broadcast_to(rr * shape[1] + cc, valid.shape).copy()\n    ids[~valid] = -1\n    return ids.astype(np.int32), valid\n\n\ndef exposure_build_geometry(gt, region_id, enforce_registered=True):\n    """Labels are used only for eligible centers and their own training targets."""\n    region_id = int(region_id)\n    if region_id not in EXPOSURE_REGIONS or gt.shape != (145,145):\n        raise ValueError("Unknown registered region or GTshape")\n    expected = EXPOSURE_REGIONS[region_id]\n    r0, r1, c0, c1 = expected["box"]\n    coords = np.argwhere(gt > 0).astype(np.int32)\n    in_box = ((coords[:,0] >= r0) & (coords[:,0] <= r1)\n              & (coords[:,1] >= c0) & (coords[:,1] <= c1))\n    test_coords = coords[in_box]\n    test_ids, test_valid = _exposure_sources(gt.shape, test_coords)\n    u_test = np.zeros(gt.size, dtype=bool)\n    u_test[test_ids[test_valid]] = True\n    train_candidates = coords[~u_test[coords[:,0] * gt.shape[1] + coords[:,1]]]\n    ids, valid = _exposure_sources(gt.shape, train_candidates)\n    s_mask = valid & u_test[np.maximum(ids, 0)]\n    r_mask = s_mask[:, ::-1, ::-1].copy()\n    eligible = ~(r_mask & (~valid | s_mask)).any(axis=(1,2))\n    train_coords = train_candidates[eligible]\n    train_ids, train_valid = ids[eligible], valid[eligible]\n    s_mask, r_mask = s_mask[eligible], r_mask[eligible]\n    if s_mask[:,5,5].any() or r_mask[:,5,5].any():\n        raise AssertionError("Intervention removes a train center")\n    blocked_keep = train_valid & ~s_mask\n    control_keep = train_valid & ~r_mask\n    blocked_union = np.zeros(gt.size, bool)\n    control_union = np.zeros(gt.size, bool)\n    blocked_union[train_ids[blocked_keep]] = True\n    control_union[train_ids[control_keep]] = True\n    exposure_counts = (test_valid & control_union[np.maximum(test_ids, 0)]).sum(axis=(1,2))\n    y_train = gt[train_coords[:,0], train_coords[:,1]].astype(np.int64) - 1\n    counts = np.bincount(y_train, minlength=16).astype(np.int64)\n    if np.any(counts < 20) or (blocked_union & u_test).any():\n        raise AssertionError("Class floor or blocked source-disjoint gate failed")\n    np.testing.assert_array_equal(blocked_keep.sum(axis=(1,2)), control_keep.sum(axis=(1,2)))\n    offsets = np.arange(-5,6)\n    shell = np.maximum(np.abs(offsets[:,None]), np.abs(offsets[None,:]))\n    for radius in range(6):\n        np.testing.assert_array_equal((s_mask & (shell == radius)).sum(axis=(1,2)),\n                                      (r_mask & (shell == radius)).sum(axis=(1,2)))\n    geometry = {"coords_train": train_coords, "y_train": y_train,\n        "counts": counts, "class_order": np.lexsort((np.arange(16),counts)).astype(np.int64),\n        "coords_test": test_coords, "test_source_union": u_test.reshape(gt.shape),\n        "s_mask": s_mask, "r_mask": r_mask, "train_source_ids": train_ids,\n        "train_valid": train_valid, "test_source_ids": test_ids, "test_valid": test_valid,\n        "blocked_source_union": blocked_union.reshape(gt.shape),\n        "control_source_union": control_union.reshape(gt.shape),\n        "exposure_counts": exposure_counts.astype(np.int64),\n        "shared_source_counts_train": s_mask.sum(axis=(1,2)).astype(np.int64),\n        "touched_train": s_mask.any(axis=(1,2))}\n    if enforce_registered:\n        assert len(train_coords) == expected["train_n"]\n        assert len(test_coords) == expected["test_n"]\n        assert int((exposure_counts > 0).sum()) == expected["affected_n"]\n        assert int((control_union & u_test).sum()) == expected["control_shared_raw_n"]\n        np.testing.assert_array_equal(counts, expected["train_counts"])\n    return geometry\n\n\ndef exposure_physical_oracle(gt, region_id, geometry):\n    """Independent nested-coordinate oracle; never calls source-ID extractor."""\n    height, width = gt.shape\n    r0,r1,c0,c1 = EXPOSURE_REGIONS[int(region_id)]["box"]\n    test_centers = [(rr,cc) for rr in range(height) for cc in range(width)\n                    if gt[rr,cc] > 0 and r0 <= rr <= r1 and c0 <= cc <= c1]\n    test_sources = set()\n    for row,col in test_centers:\n        test_sources.update((rr,cc) for rr in range(max(0,row-5),min(height,row+6))\n                            for cc in range(max(0,col-5),min(width,col+6)))\n    train_centers, oracle_s, oracle_r, counts_train = [], [], [], []\n    blocked_sources, control_sources = set(), set()\n    for row in range(height):\n        for col in range(width):\n            if gt[row,col] <= 0 or (row,col) in test_sources:\n                continue\n            patch = {(rr,cc) for rr in range(max(0,row-5),min(height,row+6))\n                     for cc in range(max(0,col-5),min(width,col+6))}\n            removed_s = patch & test_sources\n            removed_r = {(2*row-rr, 2*col-cc) for rr,cc in removed_s}\n            if any(rr < 0 or rr >= height or cc < 0 or cc >= width\n                   for rr,cc in removed_r) or removed_r & test_sources:\n                continue\n            if (row,col) in removed_s | removed_r:\n                raise AssertionError("Oracle found removed center")\n            train_centers.append((row,col))\n            sm = np.zeros((11,11),bool)\n            rm = np.zeros((11,11),bool)\n            for rr,cc in removed_s:\n                sm[rr-row+5,cc-col+5] = True\n            for rr,cc in removed_r:\n                rm[rr-row+5,cc-col+5] = True\n            oracle_s.append(sm)\n            oracle_r.append(rm)\n            counts_train.append(len(removed_s))\n            blocked_sources.update(patch - removed_s)\n            control_sources.update(patch - removed_r)\n    np.testing.assert_array_equal(np.asarray(train_centers), geometry["coords_train"])\n    np.testing.assert_array_equal(np.asarray(test_centers), geometry["coords_test"])\n    np.testing.assert_array_equal(np.asarray(oracle_s), geometry["s_mask"])\n    np.testing.assert_array_equal(np.asarray(oracle_r), geometry["r_mask"])\n    np.testing.assert_array_equal(counts_train, geometry["shared_source_counts_train"])\n    for source_set,key in ((test_sources,"test_source_union"),\n                           (blocked_sources,"blocked_source_union"),\n                           (control_sources,"control_source_union")):\n        mask = np.zeros(gt.shape,bool)\n        for rr,cc in source_set:\n            mask[rr,cc] = True\n        np.testing.assert_array_equal(mask,geometry[key])\n    oracle_e = [sum((rr,cc) in control_sources\n                    for rr in range(max(0,row-5),min(height,row+6))\n                    for cc in range(max(0,col-5),min(width,col+6)))\n                for row,col in test_centers]\n    np.testing.assert_array_equal(oracle_e,geometry["exposure_counts"])\n    assert not blocked_sources & test_sources\n    return {"independent_physical_oracle": True,\n            "blocked_shared_raw_sources": 0,\n            "control_shared_raw_sources": len(control_sources & test_sources),\n            "train_centers": len(train_centers), "test_centers": len(test_centers),\n            "affected_test_centers": int(np.count_nonzero(oracle_e))}\n\n\ndef exposure_fit_preprocessing(cube, coords_train):\n    from sklearn.decomposition import PCA\n    from sklearn.preprocessing import MinMaxScaler\n    centers = cube[coords_train[:,0],coords_train[:,1]].astype(np.float64)\n    pca = PCA(n_components=40, svd_solver="full", whiten=False)\n    pca.fit(centers)\n    projected = pca.transform(centers)\n    scaler = MinMaxScaler(feature_range=(-1,1), clip=False).fit(projected)\n    return {"pca_mean": pca.mean_, "pca_components": pca.components_,\n            "pca_explained_variance": pca.explained_variance_,\n            "scaler_scale": scaler.scale_, "scaler_min": scaler.min_,\n            "scaler_data_min": scaler.data_min_, "scaler_data_max": scaler.data_max_,\n            "fit_coords_sha256": np.asarray(exposure_sha_array(coords_train)),\n            "fit_samples": np.asarray(len(coords_train),np.int64),\n            # PCA(raw train-center mean)=0; this is its train-fitted scaled\n            # value. Image-boundary padding remains0, as in baseline B.\n            "missing_scaled_fill": scaler.min_.copy()}\n\n\ndef exposure_transform_spectra(spectra, preprocessing):\n    projected = (np.asarray(spectra,np.float64) - preprocessing["pca_mean"]) @ preprocessing["pca_components"].T\n    return projected * preprocessing["scaler_scale"] + preprocessing["scaler_min"]\n\n\ndef exposure_extract_train(cube, geometry, preprocessing, arm, indices=None):\n    if arm not in ("blocked","reuse_control"):\n        raise ValueError("Unknown arm: " + str(arm))\n    indices = np.arange(len(geometry["coords_train"])) if indices is None else np.asarray(indices,np.int64)\n    ids = geometry["train_source_ids"][indices]\n    removed = geometry["s_mask"] if arm == "blocked" else geometry["r_mask"]\n    allowed = geometry["train_valid"][indices] & ~removed[indices]\n    if arm == "blocked" and geometry["test_source_union"].ravel()[ids[allowed]].any():\n        raise AssertionError("Blocked extractor attempted to read test source")\n    # Read and transform only allowed physical coordinates, rather than build a\n    # globally transformed cube which could accidentally enter training later.\n    unique_ids, inverse = np.unique(ids[allowed], return_inverse=True)\n    transformed = exposure_transform_spectra(cube.reshape(-1,cube.shape[2])[unique_ids],preprocessing).astype(np.float32)\n    output = np.zeros((len(indices),11,11,40,1),np.float32)\n    valid_removed = geometry["train_valid"][indices] & removed[indices]\n    output[...,0][valid_removed] = preprocessing["missing_scaled_fill"].astype(np.float32)\n    output[...,0][allowed] = transformed[inverse]\n    if not np.isfinite(output).all():\n        raise ValueError("Nonfinite training observations")\n    return output\n\n\ndef exposure_counterfactual_audit(cube, gt, region_id, geometry, preprocessing):\n    """Numerical invariance of fit/support; positive control detects reuse input."""\n    changed = cube.copy()\n    changed[geometry["test_source_union"]] += 12345.6789\n    rebuilt = exposure_fit_preprocessing(changed, geometry["coords_train"])\n    for key in preprocessing:\n        np.testing.assert_array_equal(preprocessing[key],rebuilt[key])\n    # Digest every support batch in both arms. No classification is performed.\n    digests = {arm:[hashlib.sha256(),hashlib.sha256()] for arm in ("blocked","reuse_control")}\n    for start in range(0,len(geometry["coords_train"]),128):\n        indices = np.arange(start,min(start+128,len(geometry["coords_train"])))\n        for arm in digests:\n            before = exposure_extract_train(cube,geometry,preprocessing,arm,indices)\n            after = exposure_extract_train(changed,geometry,rebuilt,arm,indices)\n            digests[arm][0].update(before.tobytes())\n            digests[arm][1].update(after.tobytes())\n            if arm == "blocked":\n                np.testing.assert_array_equal(before,after)\n    hashes = {arm:[h.hexdigest() for h in pair] for arm,pair in digests.items()}\n    assert hashes["blocked"][0] == hashes["blocked"][1]\n    assert hashes["reuse_control"][0] != hashes["reuse_control"][1]\n    changed_gt = gt.copy()\n    heldout = geometry["test_source_union"] & (gt > 0)\n    changed_gt[heldout] = 1 + changed_gt[heldout] % 16\n    assert not np.array_equal(gt,changed_gt)\n    changed_geometry = exposure_build_geometry(changed_gt,region_id,enforce_registered=False)\n    for key in geometry:\n        np.testing.assert_array_equal(geometry[key],changed_geometry[key])\n    return {"preprocessing_invariant_to_test_source_spectra": True,\n            "all_blocked_train_supports_invariant": True,\n            "reuse_control_inputs_change": True,\n            "heldout_class_ids_without_eligibility_change_train_invariant": True,\n            "train_tensor_digests_original_changed": hashes,\n            "original_raw_inputs_unmodified": True}\n\n\ndef _exposure_write_train_npy(path, cube, geometry, preprocessing, arm):\n    path = Path(path)\n    temp = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")\n    try:\n        output = np.lib.format.open_memmap(temp,mode="w+",dtype=np.float32,\n                    shape=(len(geometry["coords_train"]),11,11,40,1))\n        for start in range(0,len(output),128):\n            indices = np.arange(start,min(start+128,len(output)))\n            output[start:start+len(indices)] = exposure_extract_train(cube,geometry,preprocessing,arm,indices)\n        output.flush()\n        del output\n        os.replace(temp,path)\n    finally:\n        if temp.exists():\n            temp.unlink()\n\n\ndef _exposure_selection_bytes(scientific_config):\n    text = scientific_config.get("selection_manifest_text", globals().get("EXPOSURE_SELECTION_MANIFEST_TEXT"))\n    if text is not None:\n        data = text.encode("utf-8") if isinstance(text,str) else bytes(text)\n    elif scientific_config.get("selection_manifest_path"):\n        data = Path(scientific_config["selection_manifest_path"]).read_bytes()\n    else:\n        raise ValueError("Embed the original region-selection manifest before prepare")\n    if hashlib.sha256(data).hexdigest() != EXPOSURE_SELECTION_SHA256:\n        raise ValueError("Region-selection provenance hash differs from registered grid")\n    selection = json.loads(data)\n    if selection["GT_sha256"] != EXPOSURE_RAW_SHA256["gt"] or selection["candidate_grid_starts"] != [15,45,75,105]:\n        raise ValueError("Selection provenance does not match registered GT/grid")\n    return data\n\n\ndef _exposure_verify_artifacts(root, manifest):\n    for relative,digest in manifest["artifacts"].items():\n        path = root / relative\n        if not path.is_file() or exposure_sha_file(path) != digest:\n            raise ValueError("Prepared data artifact missing/corrupt: " + relative)\n\n\ndef exposure_prepare(raw_paths, output_root, scientific_config):\n    """Create immutable-by-contract data artifacts, safely reusing hash-verified output.\n\n    Scientific configuration may contain additional training fields; data-critical\n    settings below are fixed. Metadata contains no test labels or test tensors.\n    """\n    import sklearn\n    root = Path(output_root).expanduser().resolve()\n    root.mkdir(parents=True,exist_ok=True)\n    config = dict(scientific_config)\n    if int(config.get("patch_size",11)) != 11 or int(config.get("pca_components",40)) != 40 or config.get("pca_solver","full") != "full":\n        raise ValueError("Registered preprocessing requires fullPCA40 and patch11")\n    if config.get("pca_whiten",False) or config.get("scaler_clip",False):\n        raise ValueError("Registered preprocessing disables whitening/scaler clipping")\n    if tuple(config.get("scaler_feature_range",(-1,1))) != (-1,1):\n        raise ValueError("Registered baseline B scaler uses MinMax(-1,1)")\n    selection_bytes = _exposure_selection_bytes(config)\n    for key in ("selection_manifest_text","selection_manifest_path"):\n        config.pop(key,None)\n    signature = {"raw_sha256": EXPOSURE_RAW_SHA256,\n        "selection_manifest_sha256": EXPOSURE_SELECTION_SHA256,\n        "scientific_config": config, "data_contract": "train-exposure-v2-B-scaler-mean-fill",\n        "patch_size":11, "pca_components":40, "pca_solver":"full",\n        "scaler":"MinMax(-1,1), no clipping", "image_padding_scaled":0,\n        "missing_valid_context":"scaled raw train-center mean: scaler_min",\n        "sklearn_version":sklearn.__version__, "numpy_version":np.__version__}\n    signature_hash = hashlib.sha256(_exposure_json_bytes(signature)).hexdigest()\n    manifest_path = root / "data_manifest.json"\n    if manifest_path.is_file():\n        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))\n        if manifest["signature_sha256"] != signature_hash:\n            raise ValueError("Data directory already contains different registered configuration")\n        _exposure_load_raw(raw_paths)  # Verify current mounted raw bytes too.\n        _exposure_verify_artifacts(root,manifest)\n        # This locator can change when Kaggle remounts identical raw bytes under\n        # a new slug. It is intentionally outside immutable scientific identity.\n        _exposure_atomic_bytes(root / "raw_locator.json",_exposure_json_bytes(raw_paths))\n        return {"root":str(root), "manifest":manifest, "regions":manifest["regions"]}\n    cube,gt = _exposure_load_raw(raw_paths)\n    _exposure_atomic_bytes(root / "region_selection.json",selection_bytes)\n    regions = {}\n    artifacts = {"region_selection.json":EXPOSURE_SELECTION_SHA256}\n    for region_id in sorted(EXPOSURE_REGIONS):\n        folder = root / ("region_%d" % region_id)\n        folder.mkdir(exist_ok=True)\n        geometry = exposure_build_geometry(gt,region_id)\n        oracle = exposure_physical_oracle(gt,region_id,geometry)\n        preprocessing = exposure_fit_preprocessing(cube,geometry["coords_train"])\n        counterfactual = exposure_counterfactual_audit(cube,gt,region_id,geometry,preprocessing)\n        _exposure_atomic_npz(folder / "geometry.npz",**geometry)\n        _exposure_atomic_npz(folder / "preprocessing.npz",**preprocessing)\n        for arm in ("blocked","reuse_control"):\n            _exposure_write_train_npy(folder / ("X_train_%s.npy" % arm),cube,geometry,preprocessing,arm)\n        file_hashes = {p.name:exposure_sha_file(p) for p in sorted(folder.iterdir()) if p.is_file() and p.name != "region_manifest.json" and not p.name.endswith(".tmp")}\n        metadata = {"region_id":region_id,"box_inclusive_zero_based":EXPOSURE_REGIONS[region_id]["box"],\n            "train_n":len(geometry["coords_train"]),"test_n":len(geometry["coords_test"]),\n            "affected_test_n":int((geometry["exposure_counts"] > 0).sum()),\n            "train_class_counts":geometry["counts"].tolist(),\n            "class_order_zero_based":geometry["class_order"].tolist(),\n            "touched_train_n":int(geometry["touched_train"].sum()),\n            "shared_train_source_occurrences_per_pass":int(geometry["shared_source_counts_train"].sum()),\n            "train_coords_sha256":exposure_sha_array(geometry["coords_train"]),\n            "train_labels_sha256":exposure_sha_array(geometry["y_train"]),\n            "train_class_counts_sha256":exposure_sha_array(geometry["counts"]),\n            "class_order_sha256":exposure_sha_array(geometry["class_order"]),\n            "mask_s_sha256":exposure_sha_array(geometry["s_mask"]),\n            "mask_r_sha256":exposure_sha_array(geometry["r_mask"]),\n            "preprocessing_sha256":file_hashes["preprocessing.npz"],\n            "test_exposure_counts_sha256":exposure_sha_array(geometry["exposure_counts"]),\n            "fit_scope":"same train-center spectra for both arms; outside U_test",\n            "raw_paths":dict(raw_paths),"missing_scaled_fill":preprocessing["missing_scaled_fill"].tolist(),\n            "missing_scaled_fill_sha256":exposure_sha_array(preprocessing["missing_scaled_fill"]),\n            "image_padding_scaled":0,"scaler_feature_range":[-1,1],\n            "oracle":oracle,"counterfactual":counterfactual,\n            "artifact_sha256":dict(file_hashes),"training_performed":False,\n            "test_tensor_created":False,"test_predictions_created":False}\n        _exposure_atomic_bytes(folder / "region_manifest.json",_exposure_json_bytes(metadata))\n        file_hashes["region_manifest.json"] = exposure_sha_file(folder / "region_manifest.json")\n        artifacts.update({"region_%d/%s" % (region_id,name):digest for name,digest in file_hashes.items()})\n        regions[str(region_id)] = metadata\n        print("Prepared region%d: train%d, test geometry%d, blockedrawintersection0; no test tensor." % (region_id,metadata["train_n"],metadata["test_n"]),flush=True)\n    manifest = {"signature":signature,"signature_sha256":signature_hash,\n        "raw_paths":dict(raw_paths),"regions":regions,"artifacts":artifacts,\n        "read_only_by_contract":True,"test_tensor_created":False,\n        "test_predictions_created":False,"training_performed":False}\n    _exposure_atomic_bytes(manifest_path,_exposure_json_bytes(manifest))\n    _exposure_atomic_bytes(root / "raw_locator.json",_exposure_json_bytes(raw_paths))\n    return {"root":str(root),"manifest":manifest,"regions":regions}\n\n\ndef _exposure_read_region(root,region_id):\n    root = Path(root).expanduser().resolve()\n    region_id = int(region_id)\n    if region_id not in EXPOSURE_REGIONS:\n        raise ValueError("Region must be1,2,3")\n    manifest = json.loads((root / "data_manifest.json").read_text(encoding="utf-8"))\n    folder = root / ("region_%d" % region_id)\n    metadata = manifest["regions"][str(region_id)]\n    for name in ("region_manifest.json","geometry.npz","preprocessing.npz"):\n        expected = manifest["artifacts"]["region_%d/%s" % (region_id,name)]\n        if exposure_sha_file(folder / name) != expected:\n            raise ValueError("Region artifact corrupt: " + name)\n    with np.load(folder / "geometry.npz",allow_pickle=False) as source:\n        geometry = {key:source[key] for key in source.files}\n    return root,folder,manifest,metadata,geometry\n\n\ndef exposure_load_region(root, region_id, arm):\n    if arm not in ("blocked","reuse_control"):\n        raise ValueError("Unknown arm")\n    _,folder,manifest,metadata,geometry = _exposure_read_region(root,region_id)\n    filename = "X_train_%s.npy" % arm\n    expected = manifest["artifacts"]["region_%d/%s" % (int(region_id),filename)]\n    if exposure_sha_file(folder / filename) != expected:\n        raise ValueError("Training input hash changed")\n    x_train = np.load(folder / filename,mmap_mode="r",allow_pickle=False)\n    if x_train.shape != (metadata["train_n"],11,11,40,1) or x_train.dtype != np.float32:\n        raise ValueError("Training input tensor contract differs")\n    common = {key:metadata[key] for key in ("train_coords_sha256","train_labels_sha256",\n              "train_class_counts_sha256","class_order_sha256",\n              "preprocessing_sha256","test_exposure_counts_sha256")}\n    return {"X_train":x_train,"y_train":geometry["y_train"],\n        "coords_train":geometry["coords_train"],"counts":geometry["counts"],\n        "class_order":geometry["class_order"],"touched_train":geometry["touched_train"],\n        "shared_source_counts_train":geometry["shared_source_counts_train"],\n        "metadata":dict(metadata,arm=arm,input_sha256=expected,pair_invariants=common)}\n\n\ndef exposure_load_test(root, region_id):\n    """Explicit test-stage call; returns the identical full tensor for both arms.\n\n    The orchestrator, not this data-only module, enforces suite-complete gates.\n    Test targets are materialized here, rather than exported into train artifacts.\n    The raw GT was already read for geometry/count checks during preparation.\n    """\n    root,folder,manifest,metadata,geometry = _exposure_read_region(root,region_id)\n    locator_path = root / "raw_locator.json"\n    raw_paths = (json.loads(locator_path.read_text(encoding="utf-8"))\n                 if locator_path.is_file() else manifest["raw_paths"])\n    if not all(Path(raw_paths[kind + "_path"]).is_file() for kind in ("hsi","gt")):\n        raw_paths = exposure_discover_raw(search_roots=("/kaggle/input",))\n    cube,gt = _exposure_load_raw(raw_paths)\n    with np.load(folder / "preprocessing.npz",allow_pickle=False) as source:\n        preprocessing = {key:source[key] for key in source.files}\n    ids = geometry["test_source_ids"]\n    valid = geometry["test_valid"]\n    unique_ids,inverse = np.unique(ids[valid],return_inverse=True)\n    observed = exposure_transform_spectra(cube.reshape(-1,200)[unique_ids],preprocessing).astype(np.float32)\n    x_test = np.zeros((len(ids),11,11,40,1),np.float32)\n    x_test[...,0][valid] = observed[inverse]\n    if not np.isfinite(x_test).all():\n        raise ValueError("Nonfinite full test observations")\n    coords = geometry["coords_test"]\n    y_test = gt[coords[:,0],coords[:,1]].astype(np.int64) - 1\n    return {"X_test":x_test,"y_test":y_test,"coords_test":coords,\n        "exposure_counts":geometry["exposure_counts"],\n        "metadata":dict(metadata,full_test_tensor_sha256=exposure_sha_array(x_test),\n                        test_labels_sha256=exposure_sha_array(y_test),test_tensor_created=True,\n                        same_full_test_tensor_for_both_arms=True)}\n',
    'gan_exposure_training.py': '"""HyperGAMO training for a paired source-exposure intervention.\n\nTensorFlow and tf_keras are supplied by the caller, after GPU bootstrap.  This\nmodule never imports TensorFlow, fits preprocessing, or evaluates test data.\nThe architecture, losses, batches and 34-call shared-Adam schedule follow B.\nExplicit variable lists and checkpointed dropout RNGs are a new implementation;\nthe generated masks are not a replay of the historical B TensorFlow trajectory.\n"""\nfrom pathlib import Path\nimport csv\nimport hashlib\nimport json\nimport math\nimport os\nimport time\nimport uuid\n\nimport numpy as np\n\nSCHEMA_VERSION = 1\nTRAINING_SCHEMA = "exposure_manual_tape_shared_adam_tracked_dropout_v1"\nSHAPE = (11, 11, 40, 1)\n\n\ndef _json_hash(value):\n    encoded = json.dumps(value, sort_keys=True, ensure_ascii=False,\n                         separators=(",", ":"), allow_nan=False).encode("utf-8")\n    return hashlib.sha256(encoded).hexdigest()\n\n\ndef _atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")\n    temporary.write_text(json.dumps(value, sort_keys=True, ensure_ascii=False,\n                                   indent=2, allow_nan=False) + "\\n", encoding="utf-8")\n    temporary.replace(path)\n\n\ndef _read_json(path):\n    value = json.loads(Path(path).read_text(encoding="utf-8"))\n    if not isinstance(value, dict):\n        raise ValueError("JSON record must be an object")\n    return value\n\n\ndef _file_hash(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef _array_hash(value):\n    array = np.asarray(value)\n    digest = hashlib.sha256()\n    digest.update(str(array.shape).encode("ascii"))\n    digest.update(array.dtype.str.encode("ascii"))\n    # Avoid making a second copy of the complete train tensor.\n    for start in range(0, len(array) if array.ndim else 1, 128):\n        block = array[start:start + 128] if array.ndim else array\n        digest.update(np.ascontiguousarray(block).tobytes())\n    return digest.hexdigest()\n\n\ndef _config(cfg):\n    values = {\n        "target_c_updates": int(cfg.get("target_c_updates", 8000)),\n        "batch": int(cfg.get("batch", cfg.get("batch_size", 100))),\n        "latent": int(cfg.get("latent", cfg.get("latent_dim", 100))),\n        "learning_rate": float(cfg.get("learning_rate", 2e-4)),\n        "beta1": float(cfg.get("beta1", 0.5)),\n        "gen_points": int(cfg.get("gen_points", 7)),\n        "schema": TRAINING_SCHEMA,\n        "training_implementation_sha256": cfg.get("training_implementation_sha256"),\n    }\n    if (values["target_c_updates"] != 8000 or values["batch"] != 100\n            or values["latent"] != 100 or values["gen_points"] != 7\n            or values["learning_rate"] != 2e-4 or values["beta1"] != 0.5):\n        raise ValueError("The registered experiment fixes 8000 C updates, batch100, "\n                         "latent100, 7 points/head, Adam LR2e-4 beta1=.5")\n    return values\n\n\ndef _validate_data(data):\n    X = data["X_train"]\n    y = np.asarray(data["y_train"])\n    if (np.asarray(X).dtype != np.float32 or np.asarray(X).shape != (len(y), *SHAPE)\n            or y.ndim != 1 or y.dtype.kind not in "iu" or len(y) < 100\n            or np.any(y < 0) or np.any(y > 15)):\n        raise ValueError("Train arrays must be finite float32 patches and 0-based labels")\n    for start in range(0, len(X), 256):\n        if not np.isfinite(X[start:start + 256]).all():\n            raise ValueError("Train patches contain NaN/Inf")\n    counts = np.bincount(y.astype(np.int64), minlength=16)\n    if np.any(counts == 0):\n        raise ValueError("Every training layout must retain all 16 classes")\n    order = np.argsort(counts, kind="stable")\n    if np.any(counts[order[-1]] - counts[order[:-1]] <= 0):\n        raise ValueError("The registered 15-head schedule needs one strict majority class")\n    if "counts" in data and not np.array_equal(counts, data["counts"]):\n        raise ValueError("Data class counts do not match labels")\n    if "class_order" in data and not np.array_equal(order, data["class_order"]):\n        raise ValueError("Class order must be sorted by (count, original class id)")\n    coords = np.asarray(data.get("coords_train", data.get("coords")))\n    if coords.shape != (len(y), 2) or coords.dtype.kind not in "iu":\n        raise ValueError("The ordered train coordinates are required")\n    touched = np.asarray(data.get("touched_train", np.zeros(len(y), dtype=bool)), dtype=bool)\n    shared = np.asarray(data.get("shared_source_counts_train", np.zeros(len(y), dtype=np.int64)),\n                        dtype=np.int64)\n    if (touched.shape != (len(y),) or shared.shape != (len(y),)\n            or np.any(shared < 0) or np.any(shared > 120)\n            or not np.array_equal(touched, shared > 0)):\n        raise ValueError("Source-exposure flags must agree with per-patch source counts")\n    inverse = np.empty(16, dtype=np.int64)\n    inverse[order] = np.arange(16)\n    return X, y.astype(np.int64), coords, counts, order, inverse, touched, shared\n\n\ndef _runtime(tf, keras, require_gpu=True):\n    devices = tf.config.get_visible_devices("GPU")\n    if require_gpu and not devices:\n        raise RuntimeError("This experiment requires GPU; enable GPU before running this cell")\n    if not hasattr(keras.optimizers, "legacy") or not hasattr(keras.optimizers.legacy, "Adam"):\n        raise RuntimeError("tf_keras legacy Adam is required; Keras3 is unsupported")\n    keras.mixed_precision.set_global_policy("float32")\n    keras.backend.set_image_data_format("channels_last")\n    # Failure is a gate, rather than quietly falling back to nondeterministic replay.\n    tf.config.experimental.enable_op_determinism()\n    hardware = []\n    for device in devices:\n        details = tf.config.experimental.get_device_details(device)\n        hardware.append({"device_name": str(details.get("device_name", "unknown")),\n                         "compute_capability": [int(x) for x in details.get("compute_capability", ())]})\n    build = tf.sysconfig.get_build_info()\n    return {"tensorflow": tf.__version__, "tf_keras": keras.__version__,\n            "gpu_names": [device.name for device in devices], "gpu_hardware": hardware,\n            "cuda_version": str(build.get("cuda_version", "unknown")),\n            "cudnn_version": str(build.get("cudnn_version", "unknown")),\n            "float_policy": "float32", "deterministic_ops_requested": True}\n\n\ndef _classifier(tf, keras, seed):\n    """B classifier, with an explicit trackable dropout stream."""\n    L = keras.layers\n\n    class TrackedDropout(L.Layer):\n        def __init__(self, rate, rng_seed, **kwargs):\n            super().__init__(**kwargs)\n            self.rate = float(rate)\n            self.rng = tf.random.Generator.from_seed(int(rng_seed))\n\n        def call(self, inputs, training=None):\n            def dropped():\n                uniform = self.rng.uniform(tf.shape(inputs), dtype=inputs.dtype)\n                return inputs * tf.cast(uniform >= self.rate, inputs.dtype) / (1.0 - self.rate)\n            if training is None:\n                training = False\n            if isinstance(training, (bool, np.bool_)):\n                return dropped() if training else tf.identity(inputs)\n            return tf.cond(tf.cast(training, tf.bool), dropped, lambda: tf.identity(inputs))\n\n    inputs = L.Input(shape=SHAPE)\n    x = L.Conv3D(8, (1, 1, 7), activation="relu", padding="same")(inputs)\n    x = L.Conv3D(16, (3, 3, 5), activation="relu", padding="same")(x)\n    x = L.Conv3D(32, (5, 5, 7), activation="relu", padding="same")(x)\n    s = x.shape\n    x = L.Reshape((s[1], s[2], s[3] * s[4]))(x)\n    x = L.Conv2D(64, (3, 3), activation="relu", padding="same")(x)\n    x = L.GlobalAveragePooling2D()(x)\n    dropout = TrackedDropout(0.5, seed + 101, name="paired_dropout_C")\n    x = dropout(x)\n    output = L.Dense(16, activation="softmax", kernel_initializer="he_normal")(x)\n    return keras.Model(inputs, output, name="classifier_3d_hypergamo"), dropout\n\n\ndef _build_models(tf, keras, data, cfg, seed):\n    X, y_original, _, counts_original, order, inverse, _, _ = _validate_data(data)\n    keras.backend.clear_session()\n    keras.utils.set_random_seed(int(seed))\n    L = keras.layers\n    C, c_dropout = _classifier(tf, keras, seed)\n    inputs = L.Input(shape=SHAPE)\n    x = L.Conv3D(8, (1, 1, 7), activation="relu", padding="same")(inputs)\n    x = L.Conv3D(16, (3, 3, 5), activation="relu", padding="same")(x)\n    x = L.Conv3D(32, (5, 5, 7), activation="relu", padding="same")(x)\n    s = x.shape\n    x = L.Reshape((s[1], s[2], s[3] * s[4]))(x)\n    x = L.Conv2D(64, (3, 3), activation="relu", padding="same")(x)\n    x = L.GlobalAveragePooling2D()(x)\n    label = L.Input(shape=(16,))\n    x = L.Concatenate()([x, label])\n    # Use exactly the same layer implementation, independent stream for D.\n    d_dropout = c_dropout.__class__(0.5, seed + 202, name="paired_dropout_D")\n    x = d_dropout(x)\n    output = L.Dense(1, activation="sigmoid", kernel_initializer="he_normal")(x)\n    D = keras.Model([inputs, label], output, name="D")\n\n    noise = L.Input(shape=(cfg["latent"],))\n    condition = L.Input(shape=(16,))\n    x = L.Concatenate()([noise, condition])\n    x = L.Dense(7 * 7 * 256, use_bias=False)(x)\n    x = L.BatchNormalization(momentum=0.9)(x)\n    x = L.LeakyReLU()(x)\n    x = L.Reshape((7, 7, 256))(x)\n    x = L.Conv2DTranspose(128, (5, 5), strides=(1, 1), padding="same", use_bias=False)(x)\n    x = L.BatchNormalization(momentum=0.9)(x)\n    x = L.LeakyReLU()(x)\n    x = L.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding="same", use_bias=False)(x)\n    x = L.BatchNormalization(momentum=0.9)(x)\n    x = L.LeakyReLU()(x)\n    x = L.Conv2DTranspose(1, (5, 5), strides=(2, 2), padding="same", use_bias=False)(x)\n    x = L.BatchNormalization(momentum=0.9)(x)\n    G = keras.Model([noise, condition], L.Flatten()(x), name="G")\n    y = inverse[y_original]\n    heads = []\n    for head_index in range(15):\n        support = np.ascontiguousarray(X[y == head_index].reshape(np.sum(y == head_index), -1))\n        constant = tf.constant(support, dtype=tf.float32)\n        features = L.Input(shape=(784,))\n        weights = L.Dense(len(support), activation="softmax", name="support_weights")(features)\n        # Bind each support now; do not capture the final loop constant by reference.\n        flat = L.Lambda(lambda w, support_constant=constant: tf.matmul(w, support_constant),\n                        name="convex_sum")(weights)\n        heads.append(keras.Model(features, L.Reshape(SHAPE)(flat), name=f"head_{head_index:02d}"))\n    optimizer = keras.optimizers.legacy.Adam(learning_rate=cfg["learning_rate"], beta_1=cfg["beta1"])\n    variable_groups = {"C": tuple(C.trainable_variables), "D": tuple(D.trainable_variables),\n                       "G": tuple(G.trainable_variables),\n                       "heads": [tuple(head.trainable_variables) for head in heads]}\n    all_variables = list(variable_groups["C"] + variable_groups["D"] + variable_groups["G"])\n    for group in variable_groups["heads"]:\n        all_variables.extend(group)\n    if len({id(variable) for variable in all_variables}) != len(all_variables):\n        raise RuntimeError("Trainable component variable groups unexpectedly overlap")\n    if not hasattr(optimizer, "_create_all_weights"):\n        raise RuntimeError("Legacy Adam cannot allocate all slots before checkpoint restore")\n    optimizer._create_all_weights(all_variables)\n    result = {"C": C, "D": D, "G": G, "heads": heads, "optimizer": optimizer,\n              "dropout_C": c_dropout, "dropout_D": d_dropout, "variables": variable_groups,\n              "order": order, "inverse": inverse, "y": y,\n              "counts": counts_original[order], "cfg": cfg}\n    result["updates"] = _update_functions(tf, result)\n    return result\n\n\ndef _update_functions(tf, models):\n    """Capture the exact allowed variables, independent of Keras trainable flags."""\n    C, D, G, heads = (models[key] for key in ("C", "D", "G", "heads"))\n    groups, optimizer = models["variables"], models["optimizer"]\n\n    def apply(tape, loss, variables):\n        gradients = tape.gradient(loss, variables)\n        if any(gradient is None for gradient in gradients):\n            raise RuntimeError("An active update variable is disconnected from its loss")\n        checked = [tf.debugging.check_numerics(gradient, "Nonfinite gradient") for gradient in gradients]\n        optimizer.apply_gradients(zip(checked, variables))\n        return tf.debugging.check_numerics(loss, "Nonfinite training loss")\n\n    @tf.function(reduce_retracing=True)\n    def c_step(X, labels):\n        variables = groups["C"]\n        with tf.GradientTape(watch_accessed_variables=False) as tape:\n            tape.watch(variables)\n            probability = C(X, training=True)\n            loss = tf.reduce_mean(tf.square(labels - probability))\n        return apply(tape, loss, variables)\n\n    @tf.function(reduce_retracing=True)\n    def d_step(X, labels, targets):\n        variables = groups["D"]\n        with tf.GradientTape(watch_accessed_variables=False) as tape:\n            tape.watch(variables)\n            probability = D([X, labels], training=True)\n            loss = tf.reduce_mean(tf.square(targets - probability))\n        return apply(tape, loss, variables)\n\n    gc, gd = [], []\n    for index in range(15):\n        head = heads[index]\n        variables = groups["G"] + groups["heads"][index]\n\n        def make_gc(active_head, active_variables):\n            @tf.function(reduce_retracing=True)\n            def step(noise, labels):\n                with tf.GradientTape(watch_accessed_variables=False) as tape:\n                    tape.watch(active_variables)\n                    fake = active_head(G([noise, labels], training=True), training=True)\n                    # Frozen C weights; its dropout is still in training phase,\n                    # as in the original compiled composite\'s global learning phase.\n                    probability = C(fake, training=True)\n                    loss = tf.reduce_mean(tf.square((1.0 - labels) - probability))\n                return apply(tape, loss, active_variables)\n            return step\n\n        def make_gd(active_head, active_variables):\n            @tf.function(reduce_retracing=True)\n            def step(noise, labels):\n                with tf.GradientTape(watch_accessed_variables=False) as tape:\n                    tape.watch(active_variables)\n                    fake = active_head(G([noise, labels], training=True), training=True)\n                    probability = D([fake, labels], training=True)\n                    loss = tf.reduce_mean(tf.square(1.0 - probability))\n                return apply(tape, loss, active_variables)\n            return step\n\n        gc.append(make_gc(head, variables))\n        gd.append(make_gd(head, variables))\n    return {"C": c_step, "D": d_step, "GC": gc, "GD": gd}\n\n\ndef _components(models):\n    return [(key, models[key]) for key in ("C", "D", "G")] + [\n        (f"head_{index:02d}", head) for index, head in enumerate(models["heads"])]\n\n\ndef _component_hashes(models):\n    result = {}\n    # A runtime may expose Generator.state through Layer.weights.  It is a\n    # sampling stream, not a model parameter; audit it separately below.\n    rng_variables = {id(models[key].rng.state) for key in ("dropout_C", "dropout_D")}\n    for name, model in _components(models):\n        digest = hashlib.sha256()\n        for variable in model.weights:\n            if id(variable) in rng_variables:\n                continue\n            digest.update(_array_hash(variable.numpy()).encode("ascii"))\n        result[name] = digest.hexdigest()\n    return result\n\n\ndef _optimizer_hash(models):\n    digest = hashlib.sha256()\n    for value in models["optimizer"].get_weights():\n        digest.update(_array_hash(value).encode("ascii"))\n    return digest.hexdigest()\n\n\ndef _trainable_hashes(models):\n    result = {}\n    for name, component in _components(models):\n        digest = hashlib.sha256()\n        for variable in component.trainable_variables:\n            digest.update(_array_hash(variable.numpy()).encode("ascii"))\n        result[name] = digest.hexdigest()\n    return result\n\n\ndef _check_finite(models):\n    for name, component in _components(models):\n        for variable in component.weights:\n            if not np.isfinite(variable.numpy()).all():\n                raise FloatingPointError(f"Nonfinite model variable: {name}/{variable.name}")\n    for value in models["optimizer"].get_weights():\n        if not np.isfinite(value).all():\n            raise FloatingPointError("Nonfinite shared Adam state")\n\n\ndef _dropout_state(models):\n    return {key: getattr(models[key], "rng").state.numpy().tolist()\n            for key in ("dropout_C", "dropout_D")}\n\n\nclass ExposureTrace:\n    """Arm-independent sampling with a serializable fixed-order batch cursor."""\n    def __init__(self, samples, sorted_counts, seed, region_id, state=None):\n        self.samples = int(samples)\n        self.counts = np.asarray(sorted_counts, dtype=np.int64)\n        deficits = self.counts[-1] - self.counts[:-1]\n        self.cdf = np.cumsum(deficits / deficits.sum())\n        self.cdf[-1] = 1.0\n        self.rng = np.random.default_rng(np.random.SeedSequence([int(seed), int(region_id), 314159]))\n        self.real_order = self.rng.permutation(self.samples)\n        self.loop = 0\n        self.trace_sha256 = hashlib.sha256(b"exposure_trace_v1").hexdigest()\n        self.real_counts = np.zeros(16, dtype=np.int64)\n        self.fake_counts = np.zeros(16, dtype=np.int64)\n        self.touched_real_presentations = 0\n        self.potential_shared_raw_source_occurrences = 0\n        if state is not None:\n            self.restore(state)\n\n    def restore(self, state):\n        self.rng.bit_generator.state = state["rng_state"]\n        self.real_order = np.asarray(state["real_order"], dtype=np.int64)\n        self.loop = int(state["loop"])\n        self.trace_sha256 = state["trace_sha256"]\n        self.real_counts = np.asarray(state["real_counts"], dtype=np.int64)\n        self.fake_counts = np.asarray(state["fake_counts"], dtype=np.int64)\n        self.touched_real_presentations = int(state["touched_real_presentations"])\n        self.potential_shared_raw_source_occurrences = int(state["potential_shared_raw_source_occurrences"])\n        if (self.loop < 0 or not np.array_equal(np.sort(self.real_order), np.arange(self.samples))\n                or self.real_counts.shape != (16,) or self.fake_counts.shape != (16,)\n                or np.any(self.real_counts < 0) or np.any(self.fake_counts < 0)\n                or self.fake_counts[-1] != 0 or self.touched_real_presentations < 0\n                or self.potential_shared_raw_source_occurrences < 0):\n            raise RuntimeError("Recovery sampling state is invalid")\n        expected = self.expected_presentations()\n        if self.real_counts.sum() != expected or self.fake_counts.sum() != expected:\n            raise RuntimeError("Recovery presentation counts do not match completed loops")\n\n    def expected_presentations(self):\n        batches = int(math.ceil(self.samples / 100))\n        cycles, partial = divmod(self.loop, batches)\n        return cycles * self.samples + min(partial * 100, self.samples)\n\n    def next(self):\n        batches = int(math.ceil(self.samples / 100))\n        batch_index = self.loop % batches\n        indices = self.real_order[batch_index * 100:min((batch_index + 1) * 100, self.samples)]\n        size = len(indices)\n        record = {"indices": indices,\n                  "valid": (1.0 - self.rng.uniform(0, .1, (size, 1))).astype(np.float32)}\n        record["fake_ids"] = np.searchsorted(self.cdf, self.rng.random(size), side="right")\n        record["noise"] = self.rng.normal(size=(size, 100)).astype(np.float32)\n        record["invalid"] = self.rng.uniform(0, .1, (size, 1)).astype(np.float32)\n        record["head_noise"] = [self.rng.normal(size=(7, 100)).astype(np.float32) for _ in range(15)]\n        return record\n\n    def commit(self, record, y_sorted, touched, shared, dropout_states=None):\n        self.real_counts += np.bincount(y_sorted[record["indices"]], minlength=16)\n        self.fake_counts += np.bincount(record["fake_ids"], minlength=16)\n        self.touched_real_presentations += int(touched[record["indices"]].sum())\n        self.potential_shared_raw_source_occurrences += int(shared[record["indices"]].sum())\n        digest = hashlib.sha256(bytes.fromhex(self.trace_sha256))\n        digest.update(str(self.loop).encode("ascii"))\n        for key in ("indices", "valid", "fake_ids", "noise", "invalid"):\n            digest.update(key.encode("ascii"))\n            digest.update(_array_hash(record[key]).encode("ascii"))\n        for noise in record["head_noise"]:\n            digest.update(_array_hash(noise).encode("ascii"))\n        if dropout_states is not None:\n            digest.update(_json_hash(dropout_states).encode("ascii"))\n        self.trace_sha256 = digest.hexdigest()\n        self.loop += 1\n\n    def state(self):\n        return {"rng_state": self.rng.bit_generator.state, "real_order": self.real_order.tolist(),\n                "loop": self.loop, "trace_sha256": self.trace_sha256,\n                "real_counts": self.real_counts.tolist(), "fake_counts": self.fake_counts.tolist(),\n                "touched_real_presentations": self.touched_real_presentations,\n                "potential_shared_raw_source_occurrences": self.potential_shared_raw_source_occurrences}\n\n\ndef _one_loop(tf, models, X, trace, touched, shared):\n    record = trace.next()\n    indices = record["indices"]\n    labels = np.eye(16, dtype=np.float32)[models["y"][indices]]\n    step = models["updates"]\n    losses = []\n    losses.append(float(step["C"](X[indices], labels).numpy()))\n    losses.append(float(step["D"](X[indices], labels, record["valid"]).numpy()))\n    fake_ids = record["fake_ids"]\n    fake_labels = np.eye(16, dtype=np.float32)[fake_ids]\n    # Inference generation does not update BN state or consume dropout streams.\n    features = models["G"]([record["noise"], fake_labels], training=False)\n    fake = np.empty((len(indices), *SHAPE), dtype=np.float32)\n    for head_index, head in enumerate(models["heads"]):\n        selected = np.flatnonzero(fake_ids == head_index)\n        if len(selected):\n            fake[selected] = head(tf.gather(features, selected), training=False).numpy()\n    if not np.isfinite(fake).all():\n        raise FloatingPointError("Generated training patches are nonfinite")\n    losses.append(float(step["C"](fake, fake_labels).numpy()))\n    losses.append(float(step["D"](fake, fake_labels, record["invalid"]).numpy()))\n    gc, gd = [], []\n    for head_index in range(15):\n        condition = np.zeros((7, 16), dtype=np.float32)\n        condition[:, head_index] = 1\n        noise = record["head_noise"][head_index]\n        gc.append(float(step["GC"][head_index](noise, condition).numpy()))\n        gd.append(float(step["GD"][head_index](noise, condition).numpy()))\n    values = np.asarray([(losses[0] + losses[2]) / 2,\n                         (losses[1] + losses[3]) / 2, np.mean(gc), np.mean(gd)])\n    if not np.isfinite(values).all():\n        raise FloatingPointError("Training loss is nonfinite")\n    trace.commit(record, models["y"], touched, shared, _dropout_state(models))\n    clock = int(models["optimizer"].iterations.numpy())\n    if clock != 34 * trace.loop:\n        raise RuntimeError(f"Expected 34 shared Adam calls per loop; got clock {clock}")\n    return values\n\n\ndef _checkpoint(tf, models):\n    objects = {key: models[key] for key in ("C", "D", "G")}\n    objects.update({f"head_{index:02d}": head for index, head in enumerate(models["heads"])})\n    objects["optimizer"] = models["optimizer"]\n    objects["dropout_rng_C"] = models["dropout_C"].rng\n    objects["dropout_rng_D"] = models["dropout_D"].rng\n    return tf.train.Checkpoint(**objects)\n\n\ndef _confined(jobdir, relative):\n    relative = Path(relative)\n    if relative.is_absolute() or ".." in relative.parts:\n        raise RuntimeError("Recovery references a path outside its job directory")\n    result = (Path(jobdir) / relative).resolve()\n    result.relative_to(Path(jobdir).resolve())\n    return result\n\n\ndef _restore(jobdir, checkpoint, state, contract=None):\n    if contract is not None and (state.get("contract") != contract\n                                 or state.get("contract_sha256") != _json_hash(contract)):\n        raise RuntimeError("Saved training contract does not match this job")\n    record = state.get("checkpoint", {})\n    prefix = _confined(jobdir, record.get("prefix", ""))\n    pieces = record.get("files", {})\n    expected = {path.resolve() for path in prefix.parent.glob(prefix.name + ".*")\n                if path.name == prefix.name + ".index" or path.name.startswith(prefix.name + ".data-")}\n    if (not pieces or not any(Path(relative).name == prefix.name + ".index" for relative in pieces)\n            or not any(Path(relative).name.startswith(prefix.name + ".data-") for relative in pieces)\n            or expected != {_confined(jobdir, relative) for relative in pieces}):\n        raise RuntimeError("Checkpoint index/data pieces do not match the committed manifest")\n    for relative, digest in pieces.items():\n        path = _confined(jobdir, relative)\n        if not path.is_file() or _file_hash(path) != digest:\n            raise RuntimeError(f"Checkpoint piece is missing or changed: {path}")\n    # .write/.read deliberately exclude save_counter; assert_consumed catches\n    # missing slots, head weights, BN state or explicit dropout RNGs.\n    checkpoint.read(str(prefix)).assert_consumed()\n\n\ndef _save(jobdir, checkpoint, models, state):\n    save_started = time.perf_counter()\n    _check_finite(models)\n    directory = Path(jobdir) / "recovery" / "checkpoints"\n    directory.mkdir(parents=True, exist_ok=True)\n    prefix = directory / f"u{state[\'updates\']:06d}_{uuid.uuid4().hex}"\n    checkpoint.write(str(prefix))\n    pieces = sorted(path for path in directory.glob(prefix.name + ".*")\n                    if path.name == prefix.name + ".index" or path.name.startswith(prefix.name + ".data-"))\n    if not any(path.suffix == ".index" for path in pieces) or not any(".data-" in path.name for path in pieces):\n        raise RuntimeError("TensorFlow did not write complete checkpoint pieces")\n    payload = dict(state)\n    payload["checkpoint"] = {"prefix": prefix.relative_to(jobdir).as_posix(),\n                              "files": {path.relative_to(jobdir).as_posix(): _file_hash(path)\n                                        for path in pieces}}\n    state_path = Path(jobdir) / "recovery" / "state.json"\n    previous = _read_json(state_path).get("checkpoint", {}) if state_path.is_file() else None\n    if "elapsed_seconds" in payload:\n        payload["elapsed_seconds"] = float(payload["elapsed_seconds"]) + time.perf_counter() - save_started\n    _atomic_json(state_path, payload)\n    # Only prune after publishing the complete new checkpoint.  Keep its\n    # immediate predecessor too; an interrupted write cannot invalidate resume.\n    keep = {str(prefix.resolve())}\n    if previous and previous.get("prefix"):\n        keep.add(str(_confined(jobdir, previous["prefix"])))\n    for index in directory.glob("u*.index"):\n        old_prefix = index.with_suffix("").resolve()\n        if str(old_prefix) not in keep:\n            for piece in directory.glob(old_prefix.name + ".*"):\n                if piece.name == old_prefix.name + ".index" or piece.name.startswith(old_prefix.name + ".data-"):\n                    piece.unlink()\n    return payload\n\n\ndef _history_csv(jobdir, history):\n    if not history:\n        return\n    path = Path(jobdir) / "training_history.csv"\n    temporary = path.with_name(path.name + ".tmp")\n    with temporary.open("w", encoding="utf-8", newline="") as stream:\n        writer = csv.DictWriter(stream, fieldnames=list(history[0]))\n        writer.writeheader()\n        writer.writerows(history)\n    temporary.replace(path)\n\n\ndef _data_contract(data):\n    X, y, coords, counts, order, _, touched, shared = _validate_data(data)\n    return {"train_shape": list(X.shape), "train_array_sha256": _array_hash(X),\n            "train_labels_sha256": _array_hash(y), "train_coords_sha256": _array_hash(coords),\n            "counts": counts.tolist(), "class_order": order.tolist(),\n            "touched_train_sha256": _array_hash(touched),\n            "shared_source_counts_sha256": _array_hash(shared)}\n\n\ndef _pair_data_invariants(data, data_contract):\n    """Common provenance; arm-dependent train values are deliberately excluded."""\n    metadata = data.get("metadata")\n    if not isinstance(metadata, dict):\n        raise RuntimeError("Prepared region metadata is required before pinning paired initialization")\n    common = metadata.get("pair_invariants")\n    required = ("train_coords_sha256", "train_labels_sha256", "train_class_counts_sha256",\n                "class_order_sha256", "preprocessing_sha256", "test_exposure_counts_sha256")\n    if not isinstance(common, dict) or set(common) != set(required):\n        raise RuntimeError("Prepared metadata lacks the registered common pair invariants")\n    for key in required:\n        value = common[key]\n        if (not isinstance(value, str) or len(value) != 64\n                or any(character not in "0123456789abcdef" for character in value)\n                or metadata.get(key) != value):\n            raise RuntimeError(f"Invalid or inconsistent prepared pair invariant: {key}")\n    return {\n        "actual_train_coords_sha256": data_contract["train_coords_sha256"],\n        "actual_train_labels_sha256": data_contract["train_labels_sha256"],\n        "train_shape": data_contract["train_shape"],\n        "counts": data_contract["counts"], "class_order": data_contract["class_order"],\n        "touched_train_sha256": data_contract["touched_train_sha256"],\n        "shared_source_counts_sha256": data_contract["shared_source_counts_sha256"],\n        "prepared_pair_invariants": {key: common[key] for key in required},\n    }\n\n\ndef _pair_pin_record(initialization, contract):\n    if (initialization.get("combined_sha256") != _json_hash(initialization.get("component_sha256"))\n            or initialization.get("class_order") != contract["data"]["class_order"]\n            or initialization.get("seed") != contract["seed"]\n            or initialization.get("region_id") != contract["region_id"]\n            or not isinstance(initialization.get("dropout_initial_state"), dict)\n            or set(initialization["dropout_initial_state"]) != {"dropout_C", "dropout_D"}):\n        raise RuntimeError("Initialization is incomplete or inconsistent with the paired job")\n    if not isinstance(contract.get("pair_data_invariants"), dict):\n        raise RuntimeError("Training contract does not contain common pair provenance")\n    return {"schema_version": SCHEMA_VERSION,\n            "purpose": "first arm establishes immutable paired initialization; counterpart verifies before study updates",\n            "region_id": contract["region_id"], "seed": contract["seed"],\n            "training": contract["training"], "runtime": contract["runtime"],\n            "pair_data_invariants": contract["pair_data_invariants"],\n            "initial_weights_sha256": initialization["combined_sha256"],\n            "initial_component_sha256": initialization["component_sha256"],\n            "initial_dropout_state": initialization["dropout_initial_state"],\n            "initial_dropout_state_sha256": _json_hash(initialization["dropout_initial_state"]),\n            "class_order": initialization["class_order"],\n            "readiness_barrier": False}\n\n\ndef _pair_pin_path(jobdir, contract):\n    # Both arm directories are children of this suite\'s jobs directory.\n    return (Path(jobdir).resolve().parent / "pair_initialization"\n            / f"region{int(contract[\'region_id\']):02d}_seed{int(contract[\'seed\']):02d}.json")\n\n\ndef _verify_pair_pin(jobdir, initialization, contract):\n    expected = _pair_pin_record(initialization, contract)\n    path = _pair_pin_path(jobdir, contract)\n    if not path.is_file():\n        raise RuntimeError("Paired initialization pin is missing; no study updates are authorized")\n    try:\n        saved = _read_json(path)\n    except (OSError, ValueError) as error:\n        raise RuntimeError(f"Paired initialization pin is unreadable: {path}") from error\n    if saved != expected:\n        raise RuntimeError("Paired initialization/provenance mismatch; stop before study updates")\n    return {"path": str(path), "sha256": _file_hash(path), "record_sha256": _json_hash(saved)}\n\n\ndef _pin_pair_initialization(jobdir, initialization, contract):\n    """Publish a complete immutable pin atomically, without an exists/write race.\n\n    An exclusive lock protects initial publication.  A fully-written/fsynced\n    temporary file is atomically renamed only while holding that lock and only\n    if no pin exists.  Concurrent writers either establish the same pin or\n    reject a differing winner; neither can replace it.  A crashed writer\'s\n    stale lock fails closed after a bounded wait.  The first arm need not wait\n    for its counterpart, but both check before study updates.\n    """\n    record = _pair_pin_record(initialization, contract)\n    path = _pair_pin_path(jobdir, contract)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    if path.exists():\n        return _verify_pair_pin(jobdir, initialization, contract)\n    lock_path = path.with_name(path.name + ".lock")\n    for _ in range(100):\n        try:\n            lock = lock_path.open("x", encoding="ascii")\n        except FileExistsError:\n            if path.exists():\n                return _verify_pair_pin(jobdir, initialization, contract)\n            time.sleep(0.05)\n            continue\n        temporary = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")\n        try:\n            with lock:\n                lock.write(uuid.uuid4().hex + "\\n")\n                lock.flush()\n                os.fsync(lock.fileno())\n            if not path.exists():\n                with temporary.open("x", encoding="utf-8", newline="\\n") as stream:\n                    stream.write(json.dumps(record, sort_keys=True, ensure_ascii=False,\n                                            indent=2, allow_nan=False) + "\\n")\n                    stream.flush()\n                    os.fsync(stream.fileno())\n                # All legitimate publishers hold this same exclusive lock.\n                # No existing immutable pin is ever replaced by this branch.\n                temporary.replace(path)\n            return _verify_pair_pin(jobdir, initialization, contract)\n        finally:\n            if temporary.exists():\n                temporary.unlink()\n            lock_path.unlink()\n    raise RuntimeError("Paired initialization publication is locked or incomplete; "\n                       "stop without updates and inspect the stale pin lock before retrying")\n\n\ndef exposure_train_job(tf, keras, data, jobdir, cfg, seed, region_id, arm,\n                       max_c_updates=None, save_every=250, wall_seconds=None):\n    """Train/resume a job; max_c_updates is an absolute temporary stop.\n\n    The scientific target stays 8000 in the contract.  A loop is indivisible:\n    an interruption before commit replays that loop from the previous checkpoint.\n    No test/validation calls occur here.  Return JSON-safe committed status.\n    """\n    normalized = _config(cfg)\n    runtime = _runtime(tf, keras)\n    if (arm not in ("blocked", "reuse_control") or int(region_id) not in (1, 2, 3)\n            or int(seed) not in (42, 43, 44, 45, 46)):\n        raise ValueError("Arm/region is not part of the registered experiment")\n    stop = normalized["target_c_updates"] if max_c_updates is None else int(max_c_updates)\n    if stop < 0 or stop > 8000 or stop % 2 or save_every <= 0 or save_every % 2:\n        raise ValueError("Stop and save cadence must be even; stop is within the fixed 8000-update target")\n    if wall_seconds is not None and (not np.isfinite(wall_seconds) or wall_seconds <= 0):\n        raise ValueError("wall_seconds must be a positive finite duration")\n    jobdir = Path(jobdir).resolve()\n    jobdir.mkdir(parents=True, exist_ok=True)\n    state_path = jobdir / "recovery" / "state.json"\n    if state_path.exists():\n        inspected = exposure_inspect_job(jobdir, cfg=cfg, data=data)\n        current = inspected["c_updates"]\n        # A complete marker/export is needed at target; a crash just before\n        # publishing them falls through to restore-and-finalize below.\n        if current >= stop and (current < 8000 or inspected["complete"]):\n            return {"jobdir": str(jobdir), "complete": inspected["complete"],\n                    "stop_reason": "complete" if inspected["complete"] else "already_at_requested_stop",\n                    "classifier_updates": current, "optimizer_clock": inspected["optimizer_iterations"],\n                    "initial_component_sha256": inspected["initial_weights_sha256"],\n                    "trace_sha256": inspected["trace_sha256"], "state": inspected["state"],\n                    "session_completed_loops": 0,\n                    "final_component_sha256": (_read_json(jobdir / "done.json").get("final_component_sha256")\n                                               if inspected["complete"] else None),\n                    "test_evaluated": False}\n    # Every GPU job must pass the integration gate before its first training\n    # update.  A valid cached report makes this a cheap contract/hash check.\n    smoke_dir = Path(cfg.get("gpu_smoke_dir", jobdir / "gpu_smoke"))\n    exposure_gpu_smoke(tf, keras, data, cfg, smoke_dir)\n    X, y, coords, counts, order, _, touched, shared = _validate_data(data)\n    contract = {"schema_version": SCHEMA_VERSION, "training": normalized, "runtime": runtime,\n                "seed": int(seed), "region_id": int(region_id), "arm": arm,\n                "data": _data_contract(data), "selection": "fixed final budget; no validation/test selection",\n                "schedule": {"real_batch_order": "one fixed permutation; tail retained; no epoch reshuffle",\n                             "fake_class": "majority-minus-class deficits over the 15 minority classes",\n                             "C_calls_per_loop": 2, "D_calls_per_loop": 2,\n                             "G_vs_C_calls_per_loop": 15, "G_vs_D_calls_per_loop": 15,\n                             "shared_Adam_calls_per_loop": 34},\n                "resume": "full components, Adam slots/clock, explicit dropout generators, NumPy RNG/cursor"}\n    contract["pair_data_invariants"] = _pair_data_invariants(data, contract["data"])\n    contract_path = jobdir / "training_contract.json"\n    if contract_path.is_file() and _read_json(contract_path) != contract:\n        raise RuntimeError("Existing job directory belongs to a different contract; do not overwrite it")\n    if not contract_path.exists():\n        _atomic_json(contract_path, contract)\n    models = _build_models(tf, keras, data, normalized, int(seed))\n    initial_hashes = _component_hashes(models)\n    initial_record = {"component_sha256": initial_hashes, "combined_sha256": _json_hash(initial_hashes),\n                      "seed": int(seed), "region_id": int(region_id), "class_order": order.tolist(),\n                      "fixed_supports_excluded": True, "dropout_initial_state": _dropout_state(models)}\n    initial_path = jobdir / "initialization.json"\n    if initial_path.is_file() and _read_json(initial_path) != initial_record:\n        raise RuntimeError("Fresh initialization does not match the saved job")\n    if not initial_path.exists():\n        _atomic_json(initial_path, initial_record)\n    pair_pin = _pin_pair_initialization(jobdir, initial_record, contract)\n    checkpoint = _checkpoint(tf, models)\n    trace = ExposureTrace(len(X), counts[order], seed, region_id)\n    history, recent, recent_n, elapsed_base = [], np.zeros(4), 0, 0.0\n    if state_path.exists():\n        state = _read_json(state_path)\n        _restore(jobdir, checkpoint, state, contract)\n        trace.restore(state["trace"])\n        history = list(state["history"])\n        recent = np.asarray(state["recent"], dtype=np.float64)\n        recent_n = int(state["recent_n"])\n        elapsed_base = float(state["elapsed_seconds"])\n        if (state["initial_component_sha256"] != initial_record["combined_sha256"]\n                or state["updates"] != 2 * trace.loop\n                or state["optimizer_clock"] != 34 * trace.loop\n                or int(models["optimizer"].iterations.numpy()) != 34 * trace.loop\n                or _dropout_state(models) != state["dropout_state"]\n                or recent.shape != (4,) or not np.isfinite(recent).all()\n                or recent_n < 0 or not np.isfinite(elapsed_base) or elapsed_base < 0):\n            raise RuntimeError("Recovered optimizer/RNG/history counters are inconsistent")\n    elif (jobdir / "done.json").exists():\n        raise RuntimeError("A completion marker without recovery state is not a resumable job")\n    if 2 * trace.loop > 8000:\n        raise RuntimeError("Recovery exceeds the registered target")\n    started = time.perf_counter()\n    initial_loop = trace.loop\n\n    def capture():\n        return {"schema_version": SCHEMA_VERSION, "contract": contract, "contract_sha256": _json_hash(contract),\n                "updates": 2 * trace.loop, "loop": trace.loop, "optimizer_clock": 34 * trace.loop,\n                "initial_component_sha256": initial_record["combined_sha256"], "trace": trace.state(),\n                "pair_initialization_sha256": pair_pin["sha256"],\n                "dropout_state": _dropout_state(models), "history": history,\n                "actual_shared_raw_source_occurrences_presented": (\n                    trace.potential_shared_raw_source_occurrences if arm == "reuse_control" else 0),\n                "recent": recent.tolist(), "recent_n": recent_n,\n                "elapsed_seconds": elapsed_base + time.perf_counter() - started}\n\n    if not state_path.exists():\n        state = _save(jobdir, checkpoint, models, capture())\n    stop_reason = "requested_stop"\n    last_history_loop = trace.loop\n    last_history_time = time.perf_counter()\n    session_intervals = 0\n    while 2 * trace.loop < stop:\n        loop_start = time.perf_counter()\n        recent += _one_loop(tf, models, X, trace, touched, shared)\n        recent_n += 1\n        updates = 2 * trace.loop\n        timed_out = wall_seconds is not None and time.perf_counter() - started >= wall_seconds\n        if updates % save_every == 0 or updates == stop or timed_out:\n            seconds = time.perf_counter() - last_history_time\n            completed_loops = trace.loop - last_history_loop\n            row = {"classifier_updates": updates, "optimizer_clock": 34 * trace.loop,\n                   "C_loss": float(recent[0] / recent_n), "D_loss": float(recent[1] / recent_n),\n                   "G_vs_C_loss": float(recent[2] / recent_n), "G_vs_D_loss": float(recent[3] / recent_n),\n                   "elapsed_seconds": elapsed_base + time.perf_counter() - started,\n                   "interval_seconds": seconds, "interval_loops": completed_loops,\n                   "warmup_interval": session_intervals == 0,\n                   "seconds_per_loop": seconds / completed_loops,\n                   "last_loop_seconds": time.perf_counter() - loop_start,\n                   "touched_real_presentations": trace.touched_real_presentations,\n                   "potential_shared_raw_source_occurrences": trace.potential_shared_raw_source_occurrences,\n                   "actual_shared_raw_source_occurrences_presented": (\n                       trace.potential_shared_raw_source_occurrences if arm == "reuse_control" else 0)}\n            history.append(row)\n            session_intervals += 1\n            recent[:] = 0\n            recent_n = 0\n            state = _save(jobdir, checkpoint, models, capture())\n            _history_csv(jobdir, history)\n            print(f"region{region_id} seed{seed} {arm}: {updates}/8000 C updates; "\n                  f"Adam={34 * trace.loop}; interval={seconds / completed_loops:.3f}s/loop", flush=True)\n            last_history_loop, last_history_time = trace.loop, time.perf_counter()\n        if timed_out:\n            stop_reason = "wall_seconds"\n            break\n    # Even a no-op/earlier stop reads the last atomic checkpoint, never uncommitted state.\n    state = _read_json(state_path)\n    complete = int(state["updates"]) == 8000\n    if complete:\n        _check_finite(models)\n        final_path = jobdir / "C_final.weights.h5"\n        if not final_path.exists():\n            temporary = jobdir / ("C_final." + uuid.uuid4().hex + ".weights.h5")\n            models["C"].save_weights(str(temporary))\n            temporary.replace(final_path)\n        final_record = {"complete": True, "contract_sha256": _json_hash(contract),\n                        "classifier_updates": 8000, "optimizer_clock": 136000,\n                        "initial_component_sha256": initial_record["combined_sha256"],\n                        "class_order": order.tolist(), "trace_sha256": state["trace"]["trace_sha256"],\n                        "checkpoint": state["checkpoint"], "final_classifier_sha256": _file_hash(final_path),\n                        "final_component_sha256": _component_hashes(models), "test_evaluated": False}\n        done_path = jobdir / "done.json"\n        if done_path.exists() and _read_json(done_path) != final_record:\n            raise RuntimeError("Existing final completion record does not match the checkpoint")\n        if not done_path.exists():\n            _atomic_json(done_path, final_record)\n        stop_reason = "complete"\n    status = {"jobdir": str(jobdir), "complete": complete, "stop_reason": stop_reason,\n              "classifier_updates": int(state["updates"]), "optimizer_clock": int(state["optimizer_clock"]),\n              "initial_component_sha256": initial_record["combined_sha256"],\n              "trace_sha256": state["trace"]["trace_sha256"], "state": state,\n              "session_completed_loops": trace.loop - initial_loop,\n              "final_component_sha256": _component_hashes(models) if complete else None,\n              "test_evaluated": False}\n    _atomic_json(jobdir / "status.json", {\n        "schema_version": SCHEMA_VERSION, "complete": complete, "completed": complete,\n        "c_updates": int(state["updates"]), "optimizer_iterations": int(state["optimizer_clock"]),\n        "initial_weights_sha256": initial_record["combined_sha256"],\n        "trace_sha256": state["trace"]["trace_sha256"], "stop_reason": stop_reason,\n        "checkpoint": state["checkpoint"], "recovery_state": "recovery/state.json",\n        "contract_sha256": _json_hash(contract), "test_evaluated": False})\n    del models, checkpoint\n    keras.backend.clear_session()\n    return status\n\n\ndef exposure_inspect_job(jobdir, cfg=None, data=None, require_complete=False):\n    """Read and validate committed job state and piece hashes without TensorFlow."""\n    jobdir = Path(jobdir).resolve()\n    state_path = jobdir / "recovery" / "state.json"\n    if not state_path.exists():\n        if require_complete:\n            raise RuntimeError("A required job has no committed checkpoint")\n        return {"jobdir": str(jobdir), "exists": False, "complete": False,\n                "completed": False, "c_updates": 0, "optimizer_iterations": 0}\n    state = _read_json(state_path)\n    contract = state.get("contract", {})\n    if (state.get("schema_version") != SCHEMA_VERSION\n            or state.get("contract_sha256") != _json_hash(contract)\n            or _read_json(jobdir / "training_contract.json") != contract):\n        raise RuntimeError("Committed state and job contract do not match")\n    if cfg is not None and contract.get("training") != _config(cfg):\n        raise RuntimeError("Saved job training configuration does not match")\n    if data is not None and contract.get("data") != _data_contract(data):\n        raise RuntimeError("Saved job training data does not match")\n    updates, loop = int(state.get("updates", -1)), int(state.get("loop", -1))\n    if not (0 <= updates <= 8000 and updates == 2 * loop\n            and state.get("optimizer_clock") == 34 * loop\n            and state.get("trace", {}).get("loop") == loop):\n        raise RuntimeError("Committed update/optimizer/trace counters are inconsistent")\n    data_contract = contract["data"]\n    counts = np.asarray(data_contract["counts"], dtype=np.int64)\n    order = np.asarray(data_contract["class_order"], dtype=np.int64)\n    ExposureTrace(data_contract["train_shape"][0], counts[order], contract["seed"],\n                  contract["region_id"], state=state["trace"])\n    record = state.get("checkpoint", {})\n    prefix = _confined(jobdir, record.get("prefix", ""))\n    pieces = record.get("files", {})\n    actual = {p.resolve() for p in prefix.parent.glob(prefix.name + ".*")\n              if p.name == prefix.name + ".index" or p.name.startswith(prefix.name + ".data-")}\n    if (not pieces or actual != {_confined(jobdir, relative) for relative in pieces}\n            or not any(Path(relative).name == prefix.name + ".index" for relative in pieces)\n            or not any(Path(relative).name.startswith(prefix.name + ".data-") for relative in pieces)):\n        raise RuntimeError("Checkpoint pieces do not match the committed state")\n    for relative, expected in pieces.items():\n        path = _confined(jobdir, relative)\n        if _file_hash(path) != expected:\n            raise RuntimeError(f"Checkpoint hash does not match: {path}")\n    initialization = _read_json(jobdir / "initialization.json")\n    if (initialization.get("combined_sha256") != _json_hash(initialization.get("component_sha256"))\n            or state.get("initial_component_sha256") != initialization.get("combined_sha256")):\n        raise RuntimeError("Initialization record does not match checkpoint state")\n    pair_pin = _verify_pair_pin(jobdir, initialization, contract)\n    if state.get("pair_initialization_sha256") != pair_pin["sha256"]:\n        raise RuntimeError("Paired initialization pin does not match committed checkpoint state")\n    complete = updates == 8000 and (jobdir / "done.json").exists()\n    if complete:\n        done = _read_json(jobdir / "done.json")\n        final_path = jobdir / "C_final.weights.h5"\n        if (done.get("complete") is not True or done.get("checkpoint") != record\n                or done.get("contract_sha256") != state["contract_sha256"]\n                or done.get("trace_sha256") != state["trace"]["trace_sha256"]\n                or not final_path.exists() or _file_hash(final_path) != done.get("final_classifier_sha256")):\n            raise RuntimeError("Final classifier/manifest does not match committed checkpoint")\n    if require_complete and not complete:\n        raise RuntimeError("Job is not complete at the fixed 8000-update target")\n    return {"jobdir": str(jobdir), "exists": True, "complete": complete, "completed": complete,\n            "c_updates": updates, "optimizer_iterations": 34 * loop,\n            "initial_weights_sha256": initialization["combined_sha256"],\n            "initial_component_sha256": initialization["component_sha256"],\n            "trace_sha256": state["trace"]["trace_sha256"], "checkpoint": record,\n            "class_order": order.tolist(), "state": state, "test_evaluated": False}\n\n\ndef exposure_load_classifier(tf, keras, jobdir, cfg, data=None):\n    """Restore only the committed final C; return logits in sorted class order.\n\n    Return keys: C, logits (callable), order, state.  The caller maps columns via\n    order before computing original-class metrics.  Stable NLL uses log_softmax\n    of logits, rather than clipping float32 softmax probabilities.\n    """\n    normalized = _config(cfg)\n    _runtime(tf, keras, require_gpu=False)\n    jobdir = Path(jobdir).resolve()\n    state = _read_json(jobdir / "recovery" / "state.json")\n    done = _read_json(jobdir / "done.json")\n    contract = state["contract"]\n    if (contract["training"] != normalized or done.get("complete") is not True\n            or state.get("updates") != 8000 or state.get("optimizer_clock") != 136000\n            or done.get("contract_sha256") != _json_hash(contract)\n            or done.get("checkpoint") != state.get("checkpoint")):\n        raise RuntimeError("Only the completed registered final checkpoint can be evaluated")\n    if data is not None and _data_contract(data) != contract["data"]:\n        raise RuntimeError("Classifier data contract does not match this arm")\n    final_path = jobdir / "C_final.weights.h5"\n    if not final_path.is_file() or _file_hash(final_path) != done["final_classifier_sha256"]:\n        raise RuntimeError("Exported final classifier is missing or changed")\n    keras.backend.clear_session()\n    keras.utils.set_random_seed(int(contract["seed"]))\n    C, dropout = _classifier(tf, keras, int(contract["seed"]))\n    C.load_weights(str(final_path))\n    dense = C.layers[-1]\n    feature_model = keras.Model(C.input, dense.input)\n\n    def logits(inputs, training=False):\n        if training:\n            raise ValueError("The evaluation logits accessor is inference-only")\n        features = feature_model(inputs, training=False)\n        return tf.linalg.matmul(features, dense.kernel) + dense.bias\n\n    probe = np.zeros((2, *SHAPE), dtype=np.float32)\n    np.testing.assert_allclose(tf.nn.softmax(logits(probe)).numpy(), C(probe, training=False).numpy(),\n                               rtol=1e-6, atol=1e-7)\n    order = np.asarray(contract["data"]["class_order"], dtype=np.int64)\n    if not np.array_equal(np.sort(order), np.arange(16)):\n        raise RuntimeError("Saved class order is invalid")\n    return {"C": C, "classifier": C, "logits": logits, "order": order,\n            "class_order": order, "state": state}\n\n\ndef exposure_gpu_smoke(tf, keras, region_data, cfg, out):\n    """Mandatory GPU integration gate: frozen routes and exact resume replay."""\n    normalized = _config(cfg)\n    runtime = _runtime(tf, keras)\n    X, y, _, counts, order, _, touched, shared = _validate_data(region_data)\n    out = Path(out).resolve()\n    out.mkdir(parents=True, exist_ok=True)\n    contract = {"training": normalized, "runtime": runtime, "counts": counts.tolist(),\n                "class_order": order.tolist(), "data": _data_contract(region_data)}\n    report_path = out / "gpu_smoke.json"\n    if report_path.exists():\n        report = _read_json(report_path)\n        if report.get("passed") is True and report.get("contract") == contract:\n            return report\n        raise RuntimeError("Existing GPU smoke report belongs to another contract or did not pass")\n    seed, region_id = 42, 1\n    models = _build_models(tf, keras, region_data, normalized, seed)\n    initial_hashes = _component_hashes(models)\n    initial_trainable_hashes = _trainable_hashes(models)\n    labels = np.eye(16, dtype=np.float32)[models["y"][:7]]\n    noise = np.random.default_rng(991).normal(size=(7, 100)).astype(np.float32)\n    freeze_checks = []\n\n    def assert_route(route, allowed, action):\n        before = _component_hashes(models)\n        trainable_before = _trainable_hashes(models)\n        rng_before = _dropout_state(models)\n        clock = int(models["optimizer"].iterations.numpy())\n        action()\n        after = _component_hashes(models)\n        trainable_after = _trainable_hashes(models)\n        rng_after = _dropout_state(models)\n        changed = {key for key in before if before[key] != after[key]}\n        changed_trainables = {key for key in trainable_before\n                              if trainable_before[key] != trainable_after[key]}\n        if changed != set(allowed):\n            raise RuntimeError(f"Frozen-variable smoke failed for {route}: changed={sorted(changed)}")\n        if changed_trainables != set(allowed):\n            raise RuntimeError(f"Trainable-variable smoke failed for {route}: {sorted(changed_trainables)}")\n        if int(models["optimizer"].iterations.numpy()) != clock + 1:\n            raise RuntimeError("One update route did not make exactly one shared Adam call")\n        expected_rng = "dropout_C" if route == "C" or route.startswith("GC_") else "dropout_D"\n        changed_rng = {key for key in rng_before if rng_before[key] != rng_after[key]}\n        if changed_rng != {expected_rng}:\n            raise RuntimeError(f"Dropout training-phase smoke failed for {route}: {changed_rng}")\n        freeze_checks.append({"route": route, "changed_components": sorted(changed),\n                              "changed_trainable_components": sorted(changed_trainables),\n                              "allowed_components": list(allowed),\n                              "advanced_dropout_stream": expected_rng, "passed": True})\n\n    assert_route("C", ["C"], lambda: models["updates"]["C"](X[:7], labels).numpy())\n    assert_route("D", ["D"], lambda: models["updates"]["D"](\n        X[:7], labels, np.ones((7, 1), dtype=np.float32)).numpy())\n    for head_index in (0, 14):\n        condition = np.zeros((7, 16), dtype=np.float32)\n        condition[:, head_index] = 1\n        assert_route(f"GC_{head_index}", ["G", f"head_{head_index:02d}"],\n                     lambda: models["updates"]["GC"][head_index](noise, condition).numpy())\n        assert_route(f"GD_{head_index}", ["G", f"head_{head_index:02d}"],\n                     lambda: models["updates"]["GD"][head_index](noise, condition).numpy())\n    inference_before = {"weights": _component_hashes(models), "rng": _dropout_state(models),\n                        "clock": int(models["optimizer"].iterations.numpy())}\n    features = models["G"]([noise, labels], training=False)\n    for head in models["heads"]:\n        value = head(features, training=False).numpy()\n        if value.shape != (7, *SHAPE) or not np.isfinite(value).all():\n            raise RuntimeError("Inference generation smoke produced invalid patches")\n    inference_after = {"weights": _component_hashes(models), "rng": _dropout_state(models),\n                       "clock": int(models["optimizer"].iterations.numpy())}\n    if inference_before != inference_after:\n        raise RuntimeError("Detached generation changed model/BN/RNG/optimizer state")\n    del models\n    keras.backend.clear_session()\n    # Fresh networks after route checks: smoke updates never enter a training job.\n    models = _build_models(tf, keras, region_data, normalized, seed)\n    if _component_hashes(models) != initial_hashes:\n        raise RuntimeError("Fresh initialization is not deterministic")\n    trace = ExposureTrace(len(X), counts[order], seed, region_id)\n    first_start = time.perf_counter()\n    _one_loop(tf, models, X, trace, touched, shared)\n    after_full_loop = _trainable_hashes(models)\n    if any(after_full_loop[key] == initial_trainable_hashes[key] for key in initial_trainable_hashes):\n        raise RuntimeError("The complete loop did not update every component, including all 15 heads")\n    first_seconds = time.perf_counter() - first_start\n    checkpoint = _checkpoint(tf, models)\n    smoke_state = {"updates": 2, "contract": contract, "contract_sha256": _json_hash(contract),\n                   "trace": trace.state(), "dropout_state": _dropout_state(models)}\n    state = _save(out, checkpoint, models, smoke_state)\n    second_start = time.perf_counter()\n    loss_uninterrupted = _one_loop(tf, models, X, trace, touched, shared)\n    second_seconds = time.perf_counter() - second_start\n    expected = {"components": _component_hashes(models), "optimizer": _optimizer_hash(models),\n                "dropout": _dropout_state(models), "trace": trace.state()}\n    del models, checkpoint\n    keras.backend.clear_session()\n    models = _build_models(tf, keras, region_data, normalized, seed)\n    checkpoint = _checkpoint(tf, models)\n    _restore(out, checkpoint, state, contract)\n    trace = ExposureTrace(len(X), counts[order], seed, region_id, state=state["trace"])\n    if _dropout_state(models) != state["dropout_state"]:\n        raise RuntimeError("Checkpoint did not restore both dropout generator states")\n    loss_resumed = _one_loop(tf, models, X, trace, touched, shared)\n    actual = {"components": _component_hashes(models), "optimizer": _optimizer_hash(models),\n              "dropout": _dropout_state(models), "trace": trace.state()}\n    if expected != actual or not np.array_equal(loss_uninterrupted, loss_resumed):\n        raise RuntimeError("GPU checkpoint continuation was not bitwise reproducible; do not train")\n    report = {"passed": True, "contract": contract, "frozen_variable_routes": freeze_checks,\n              "fresh_initialization_sha256": _json_hash(initial_hashes),\n              "all_15_heads_exercised": True, "shared_Adam_calls_per_loop": 34,\n              "resume_exact_component_and_optimizer_hashes": True,\n              "resume_exact_NumPy_and_dropout_streams": True,\n              "inference_generation_keeps_BN_RNG_optimizer_unchanged": True,\n              "optimizer_clock_after_two_loops": int(models["optimizer"].iterations.numpy()),\n              "first_loop_with_compilation_seconds": first_seconds,\n              "second_loop_seconds": second_seconds,\n              "smoke_updates_are_not_training_budget": True, "test_evaluated": False}\n    _atomic_json(report_path, report)\n    del models, checkpoint\n    keras.backend.clear_session()\n    return report\n',
    'gan_exposure_runner.py': '"""Standalone Kaggle orchestration for the registered HyperGAMO exposure suite.\n\nThe release builder embeds this module and its two helpers in one notebook cell.\nNo training runs merely by importing this file. GPU workers use fresh processes.\n"""\nfrom pathlib import Path\nimport csv\nimport hashlib\nimport json\nimport os\nimport shutil\nimport subprocess\nimport sys\nimport time\nimport zipfile\n\n\ndef _exposure_json_hash(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=True,\n                                    separators=(",", ":"), allow_nan=False).encode()).hexdigest()\n\n\ndef _exposure_file_hash(path):\n    h = hashlib.sha256()\n    with Path(path).open("rb") as f:\n        for block in iter(lambda: f.read(1024 * 1024), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef _exposure_write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + ".tmp")\n    tmp.write_text(json.dumps(value, sort_keys=True, ensure_ascii=False,\n                              indent=2, allow_nan=False) + "\\n", encoding="utf-8")\n    tmp.replace(path)\n\n\ndef _exposure_read_json(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef _exposure_release_config():\n    from gan_exposure_release import SCIENTIFIC_CONFIG, RELEASE_ID, COMPONENT_HASHES\n    cfg = json.loads(json.dumps(SCIENTIFIC_CONFIG))\n    cfg["implementation_release"] = RELEASE_ID\n    cfg["component_sha256"] = COMPONENT_HASHES\n    return cfg\n\n\ndef _exposure_expected_jobs():\n    return [(region, seed, arm) for region in (1, 2, 3)\n            for seed in range(42, 47) for arm in ("blocked", "reuse_control")]\n\n\ndef _exposure_job_id(region, seed, arm):\n    if (region, seed, arm) not in _exposure_expected_jobs():\n        raise ValueError("Job nằm ngoài ma trận 30 lượt đã khóa.")\n    return f"region{region:02d}_seed{seed}_{arm}"\n\n\ndef _exposure_suite_spec(cfg):\n    return {"schema": "gan-train-exposure-suite-v1", "scientific_config": cfg,\n            "jobs": [{"region": r, "seed": s, "arm": a,\n                      "job_id": _exposure_job_id(r, s, a)}\n                     for r, s, a in _exposure_expected_jobs()],\n            "scope": "HyperGAMO system; fixed full test; not isolated G or B inflation",\n            "test_gate": "all30_complete_and_pairs_valid; no scoring during benchmark/train"}\n\n\ndef _exposure_validate_suite(root, cfg):\n    p = Path(root) / "suite_manifest.json"\n    doc = _exposure_read_json(p)\n    expected = _exposure_suite_spec(cfg)\n    if doc.get("spec") != expected or doc.get("suite_id") != _exposure_json_hash(expected)[:12]:\n        raise RuntimeError(f"Suite/config/release không khớp: {p}")\n    return doc\n\n\ndef _exposure_snapshot_fingerprint(root):\n    # Different saved snapshots must be selected explicitly, not silently merged.\n    rows = []\n    for p in sorted((Path(root) / "jobs").glob("*/recovery/state.json")):\n        rows.append((p.relative_to(root).as_posix(), _exposure_file_hash(p)))\n    for p in sorted((Path(root) / "jobs").glob("*/done.json")):\n        rows.append((p.relative_to(root).as_posix(), _exposure_file_hash(p)))\n    return _exposure_json_hash(rows)\n\n\ndef _exposure_restore_suite(output_root, cfg, explicit=None):\n    spec = _exposure_suite_spec(cfg)\n    suite_id = _exposure_json_hash(spec)[:12]\n    destination = Path(output_root) / f"KLTN_GAN_Exposure_{suite_id}"\n    if destination.exists():\n        _exposure_validate_suite(destination, cfg)\n        if explicit and Path(explicit).resolve() != destination.resolve():\n            raise RuntimeError("Working đã có suite. Dùng suite hiện tại hoặc chọn Output root khác "\n                               "để khôi phục snapshot riêng; không ghi đè checkpoint.")\n        return destination\n    candidates = []\n    if explicit:\n        candidate = Path(explicit)\n        _exposure_validate_suite(candidate, cfg)\n        candidates = [candidate]\n    elif Path("/kaggle/input").is_dir():\n        for p in Path("/kaggle/input").rglob("suite_manifest.json"):\n            if p.parent.name == destination.name:\n                _exposure_validate_suite(p.parent, cfg)\n                candidates.append(p.parent)\n    if candidates:\n        fingerprints = {_exposure_snapshot_fingerprint(p) for p in candidates}\n        if len(fingerprints) != 1:\n            raise RuntimeError("Có nhiều snapshot khác tiến độ. Điền RESUME_RUN_PATH đúng thư mục "\n                               "suite muốn tiếp tục:\\n" + "\\n".join(map(str, candidates)))\n        source = sorted(candidates, key=str)[0]\n        print("Khôi phục Output đã lưu:", source, flush=True)\n        shutil.copytree(source, destination)\n        _exposure_validate_suite(destination, cfg)\n    else:\n        destination.mkdir(parents=True)\n        _exposure_write_json(destination / "suite_manifest.json",\n                             {"suite_id": suite_id, "spec": spec,\n                              "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())})\n    return destination\n\n\ndef _exposure_gpu_inventory():\n    try:\n        out = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total",\n                              "--format=csv,noheader,nounits"],\n                             capture_output=True, text=True, check=True, timeout=20).stdout\n        rows = []\n        for line in out.strip().splitlines():\n            index, name, memory = (part.strip() for part in line.split(",", 2))\n            rows.append({"index": int(index), "name": name, "memory_mb": int(memory)})\n        if rows:\n            return rows\n    except (OSError, ValueError, subprocess.SubprocessError):\n        pass\n    raise RuntimeError("Chưa có GPU. Kaggle Settings > Accelerator > GPU T4/P100, "\n                       "rồi chạy riêng cell này. Không train bằng CPU.")\n\n\ndef _exposure_bootstrap_gpu():\n    # Called inside a fresh worker with a single CUDA_VISIBLE_DEVICES value.\n    os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")\n    os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")\n    os.environ.setdefault("TF_DETERMINISTIC_OPS", "1")\n    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")\n    import tensorflow as tf\n    try:\n        import tf_keras as keras\n    except ImportError as e:\n        raise RuntimeError("Môi trường thiếu tf_keras. Cần tf_keras cùng minor version TensorFlow; "\n                           "runner không tự đổi/install thư viện trong lượt train.") from e\n    if tf.__version__.split(".")[:2] != keras.__version__.split(".")[:2]:\n        raise RuntimeError(f"TensorFlow/tf_keras lệch version: {tf.__version__}/{keras.__version__}")\n    devices = tf.config.list_physical_devices("GPU")\n    if len(devices) != 1:\n        raise RuntimeError(f"Worker phải nhìn đúng một GPU, hiện thấy {len(devices)}.")\n    for gpu in devices:\n        tf.config.experimental.set_memory_growth(gpu, True)\n    tf.config.experimental.enable_op_determinism()\n    print(f"Worker GPU: TF {tf.__version__}, tf_keras {keras.__version__}; "\n          f"CUDA_VISIBLE_DEVICES={os.environ.get(\'CUDA_VISIBLE_DEVICES\')}", flush=True)\n    return tf, keras\n\n\ndef _exposure_inspect_job(jobdir, cfg, data=None, require_complete=False):\n    from gan_exposure_training import exposure_inspect_job\n    record = exposure_inspect_job(jobdir, cfg, data=data, require_complete=require_complete)\n    if record.get("exists"):\n        contract = record["state"]["contract"]\n        data_contract = contract["data"]\n        record["train_coords_sha256"] = data_contract["train_coords_sha256"]\n        record["train_labels_sha256"] = data_contract["train_labels_sha256"]\n        record["common_config_sha256"] = _exposure_json_hash(\n            {"release_config": cfg, "training": contract["training"], "runtime": contract["runtime"],\n             "region_id": contract["region_id"], "seed": contract["seed"]})\n        if data is not None:\n            record["preprocessing_sha256"] = data["metadata"]["preprocessing_sha256"]\n        history = record["state"].get("history", [])\n        steady = [row for row in history if not row.get("warmup_interval", False)\n                  and row.get("classifier_updates", 0) > 100\n                  and row.get("interval_loops", 0) > 0]\n        if steady:\n            record["seconds_per_c_update"] = (\n                sum(row["interval_seconds"] for row in steady)\n                / (2 * sum(row["interval_loops"] for row in steady)))\n    return record\n\n\ndef _exposure_pair_gate(root, cfg, region, seed, require_complete=False):\n    from gan_exposure_data import exposure_load_region\n    records = []\n    for arm in ("blocked", "reuse_control"):\n        data = exposure_load_region(Path(root) / "data", region, arm)\n        jobdir = Path(root) / "jobs" / _exposure_job_id(region, seed, arm)\n        records.append(_exposure_inspect_job(jobdir, cfg, data, require_complete))\n    # The training helper defines pair-invariant metadata and active trace prefixes.\n    left, right = records\n    if not all(record.get("exists") for record in records):\n        raise RuntimeError("Pair chưa có đủ hai checkpoint đã commit.")\n    for key in ("initial_weights_sha256", "train_coords_sha256", "train_labels_sha256",\n                "preprocessing_sha256", "class_order", "common_config_sha256"):\n        if left.get(key) is None or left.get(key) != right.get(key):\n            raise RuntimeError(f"Pair region {region}/seed {seed} không khớp {key}.")\n    if left.get("c_updates") == right.get("c_updates"):\n        if left.get("trace_sha256") is None or left.get("trace_sha256") != right.get("trace_sha256"):\n            raise RuntimeError(f"Pair region {region}/seed {seed} khác trace cùng tiến độ.")\n    return records\n\n\ndef _exposure_all_test_gate(root, cfg):\n    _exposure_pin_data(root)\n    rows = []\n    for region in (1, 2, 3):\n        for seed in range(42, 47):\n            pair = _exposure_pair_gate(root, cfg, region, seed, require_complete=True)\n            for arm, state in zip(("blocked", "reuse_control"), pair):\n                if state.get("c_updates") != cfg["target_c_updates"]:\n                    raise RuntimeError("Test chỉ mở khi mọi job đủ đúng ngân sách đã khóa.")\n                rows.append({"region": region, "seed": seed, "arm": arm, **state})\n    return rows\n\n\ndef _exposure_pin_data(root):\n    path = Path(root) / "data" / "data_manifest.json"\n    digest = _exposure_file_hash(path)\n    pin = Path(root) / "data_certificate_pin.json"\n    record = {"data_manifest_sha256": digest}\n    if pin.exists():\n        if _exposure_read_json(pin) != record:\n            raise RuntimeError("Data manifest khác chứng nhận đã khóa cho suite.")\n    else:\n        _exposure_write_json(pin, record)\n    return record\n\n\ndef _exposure_report_zip(root):\n    root = Path(root)\n    out = root.parent / (root.name + "_REPORT.zip")\n    tmp = out.with_suffix(".zip.tmp")\n    # Small reports, no model weights or train tensors.\n    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:\n        for p in sorted(root.rglob("*")):\n            if (p.is_file() and p.suffix in {".json", ".csv", ".md"}\n                    and "checkpoints" not in p.parts and "runner" not in p.parts):\n                z.write(p, p.relative_to(root.parent).as_posix())\n    tmp.replace(out)\n    return out\n\n\ndef _exposure_training_summary(root, cfg):\n    rows = []\n    for region, seed, arm in _exposure_expected_jobs():\n        jobdir = Path(root) / "jobs" / _exposure_job_id(region, seed, arm)\n        try:\n            state = _exposure_inspect_job(jobdir, cfg)\n            rows.append({"region": region, "seed": seed, "arm": arm,\n                         "c_updates": state.get("c_updates", 0),\n                         "complete": state.get("completed", False),\n                         "optimizer_iterations": state.get("optimizer_iterations", 0),\n                         "seconds_per_c_update": state.get("seconds_per_c_update")})\n        except FileNotFoundError:\n            rows.append({"region": region, "seed": seed, "arm": arm,\n                         "c_updates": 0, "complete": False, "optimizer_iterations": 0,\n                         "seconds_per_c_update": None})\n    _exposure_write_json(Path(root) / "training_progress.json",\n                         {"jobs": rows, "test_scored": False})\n    timings = [row["seconds_per_c_update"] for row in rows\n               if isinstance(row.get("seconds_per_c_update"), (int, float))\n               and row["seconds_per_c_update"] > 0]\n    if timings:\n        seconds = sum(timings) / len(timings)\n        remaining = sum(cfg["target_c_updates"] - row["c_updates"] for row in rows)\n        estimate = {"mean_measured_seconds_per_c_update": seconds,\n                    "remaining_c_updates": remaining,\n                    "estimated_remaining_gpu_hours": remaining * seconds / 3600,\n                    "scope": "measured training throughput only; checkpoint/startup variance excluded; "\n                             "new regions can differ; no rental price assumed",\n                    "benchmark_jobs_used": len(timings)}\n        _exposure_write_json(Path(root) / "benchmark_summary.json", estimate)\n        print("Ước lượng từ tốc độ thực:", round(estimate["estimated_remaining_gpu_hours"], 2),\n              "GPU-hours còn lại (ước lượng).", flush=True)\n    print("Tiến độ:", sum(bool(x["complete"]) for x in rows), "/30 job hoàn tất;",\n          sum(int(x["c_updates"]) for x in rows), "/240000 C updates.", flush=True)\n    return rows\n\n\ndef _exposure_pump_log(path, offset):\n    path = Path(path)\n    if not path.is_file():\n        return offset\n    with path.open("r", encoding="utf-8", errors="replace") as f:\n        f.seek(offset)\n        text = f.read()\n        offset = f.tell()\n    if text:\n        print(text, end="", flush=True)\n    return offset\n\n\ndef _exposure_worker_queues(jobs, gpu_count):\n    if gpu_count < 1:\n        raise ValueError("Cần ít nhất một GPU worker.")\n    queues = [[] for _ in range(gpu_count)]\n    pairs = {(region, seed): i for i, (region, seed) in enumerate(\n        (region, seed) for region in (1, 2, 3) for seed in range(42, 47))}\n    for i, job in enumerate(jobs):\n        region, seed, arm = job\n        if arm == "test":\n            lane = i % gpu_count\n        else:\n            pair_index = pairs[(region, seed)]\n            arm_index = ("blocked", "reuse_control").index(arm)\n            # Alternate arms across devices for successive registered pairs;\n            # filtering or resuming jobs does not change their assignment.\n            lane = (pair_index + arm_index) % gpu_count\n        queues[lane].append(list(job))\n    return queues\n\n\ndef _exposure_run_workers(root, cfg, jobs, gpu_ids, code_root, runtime, test=False):\n    queues = _exposure_worker_queues(jobs, len(gpu_ids))\n    children = []\n    for worker_id, (gpu, queue) in enumerate(zip(gpu_ids, queues)):\n        if not queue:\n            continue\n        log_path = Path(root) / f"worker_{runtime[\'action\']}_{worker_id}.log"\n        log_stream = log_path.open("w", encoding="utf-8")\n        env = os.environ.copy()\n        env["CUDA_VISIBLE_DEVICES"] = str(gpu)\n        env["TF_USE_LEGACY_KERAS"] = "1"\n        env["TF_DETERMINISTIC_OPS"] = "1"\n        env["PYTHONUNBUFFERED"] = "1"\n        env["EXPOSURE_WORKER_TASK"] = json.dumps({"root": str(root), "jobs": queue,\n                                                 "runtime": runtime, "test": test})\n        launcher = ("import sys; sys.path.insert(0, " + repr(str(code_root)) + "); "\n                    "from gan_exposure_runner import exposure_worker_entry; exposure_worker_entry()")\n        child = subprocess.Popen([sys.executable, "-u", "-c", launcher], env=env,\n                                 stdout=log_stream, stderr=subprocess.STDOUT)\n        children.append({"process": child, "log": log_path, "stream": log_stream,\n                         "offset": 0, "gpu": gpu})\n    try:\n        last_heartbeat = time.monotonic()\n        while any(x["process"].poll() is None for x in children):\n            for item in children:\n                item["offset"] = _exposure_pump_log(item["log"], item["offset"])\n            failed = [x for x in children if x["process"].poll() not in (None, 0)]\n            if failed:\n                for x in failed:\n                    x["offset"] = _exposure_pump_log(x["log"], x["offset"])\n                raise RuntimeError("Worker lỗi; xem log bên trên. Checkpoint đã commit được giữ. "\n                                   + ", ".join(str(x["log"]) for x in failed))\n            if time.monotonic() - last_heartbeat >= 30:\n                active = [x["gpu"] for x in children if x["process"].poll() is None]\n                print("GPU worker còn chạy:", active, "| checkpoint tự lưu.", flush=True)\n                last_heartbeat = time.monotonic()\n            time.sleep(2)\n        for x in children:\n            x["offset"] = _exposure_pump_log(x["log"], x["offset"])\n            if x["process"].returncode != 0:\n                raise RuntimeError(f"Worker exit {x[\'process\'].returncode}: {x[\'log\']}")\n    finally:\n        for x in children:\n            # Stop spawned helpers on interruption; never leave an unseen GPU job.\n            if x["process"].poll() is None:\n                x["process"].terminate()\n                try:\n                    x["process"].wait(timeout=20)\n                except subprocess.TimeoutExpired:\n                    x["process"].kill()\n                    x["process"].wait(timeout=10)\n            x["stream"].close()\n\n\ndef exposure_launch(options, code_root):\n    from gan_exposure_data import exposure_discover_raw, exposure_prepare\n    from gan_exposure_release import SELECTION_MANIFEST_TEXT\n    cfg = _exposure_release_config()\n    action = options.get("action", "benchmark")\n    if action not in {"prepare", "benchmark", "train", "test"}:\n        raise ValueError("ACTION phải là prepare, benchmark, train hoặc test.")\n    root = _exposure_restore_suite(options.get("output_root", "/kaggle/working"), cfg,\n                                   options.get("resume_run_path"))\n    print("Suite:", root, flush=True)\n    source = exposure_discover_raw(options.get("hsi_path"), options.get("gt_path"))\n    data_cfg = dict(cfg, selection_manifest_text=SELECTION_MANIFEST_TEXT)\n    exposure_prepare(source, root / "data", data_cfg)\n    _exposure_pin_data(root)\n    if action == "prepare":\n        print("CPU audit xong. Chưa train hoặc chấm test.", flush=True)\n        print("ZIP báo cáo:", _exposure_report_zip(root), flush=True)\n        return str(root)\n    inventory = _exposure_gpu_inventory()\n    wanted = options.get("gpu_ids")\n    gpu_ids = [row["index"] for row in inventory] if wanted is None else list(wanted)\n    if not gpu_ids or len(gpu_ids) != len(set(gpu_ids)) or not set(gpu_ids) <= {x["index"] for x in inventory}:\n        raise ValueError("GPU_IDS không thuộc danh sách GPU hiện có.")\n    gpu_ids = gpu_ids[:int(options.get("max_workers", 2))]\n    if not gpu_ids:\n        raise ValueError("MAX_WORKERS phải >=1.")\n    selected_names = {x["name"] for x in inventory if x["index"] in gpu_ids}\n    if len(selected_names) != 1:\n        raise RuntimeError("GPU được chọn khác model. Chọn GPU_IDS=[0] hoặc một nhóm GPU cùng model "\n                           "để tránh khác hardware giữa hai nhánh ghép cặp.")\n    if action == "test":\n        _exposure_all_test_gate(root, cfg)  # Before loading any test tensor/model.\n        _exposure_run_workers(root, cfg, [(0, 0, "test")], gpu_ids[:1], code_root,\n                              {"action": action}, test=True)\n        print("TEST XONG — xem paired_summary.csv và region_summary.csv.", flush=True)\n    else:\n        if action == "benchmark":\n            jobs = [(1, 42, "blocked"), (1, 42, "reuse_control")]\n            stop = int(options.get("benchmark_c_updates", 400))\n            if stop < 100 or stop > cfg["target_c_updates"] or stop % 2:\n                raise ValueError("BENCHMARK_C_UPDATES là số chẵn 100..8000.")\n        else:\n            jobs = _exposure_expected_jobs()\n            stop = cfg["target_c_updates"]\n            if options.get("job_filter") is not None:\n                selected = [tuple(x) for x in options["job_filter"]]\n                if not selected or len(selected) != len(set(selected)) or any(x not in jobs for x in selected):\n                    raise ValueError("JOB_FILTER chứa job ngoài suite hoặc trùng nhau.")\n                jobs = [x for x in jobs if x in selected]\n        runtime = {"action": action, "stop_c_updates": stop,\n                   "session_seconds": float(options.get("session_hours", 8)) * 3600,\n                   "save_every": int(options.get("save_every", 100))}\n        if runtime["session_seconds"] <= 0 or runtime["save_every"] < 2 or runtime["save_every"] % 2:\n            raise ValueError("SESSION_HOURS phải >0; SAVE_EVERY là số chẵn >=2.")\n        print(f"{action}: {len(jobs)} jobs, {len(gpu_ids)} GPU worker độc lập; "\n              f"stop {stop} C updates/job. Không chấm test.", flush=True)\n        try:\n            _exposure_run_workers(root, cfg, jobs, gpu_ids, code_root, runtime)\n        finally:\n            _exposure_training_summary(root, cfg)\n            print("ZIP báo cáo:", _exposure_report_zip(root), flush=True)\n        # Prefix trace equality is testable only at matching stop counts.\n        for region in (1, 2, 3):\n            for seed in range(42, 47):\n                if all((root / "jobs" / _exposure_job_id(region, seed, arm) / "recovery" / "state.json").exists()\n                       for arm in ("blocked", "reuse_control")):\n                    _exposure_pair_gate(root, cfg, region, seed)\n        if action == "benchmark":\n            print(f"BENCHMARK XONG. Đổi ACTION=\'train\' để tiếp tục checkpoint của cùng suite; "\n                  f"{stop} bước đầu đã nằm trong ngân sách 8000.", flush=True)\n        else:\n            print("Lưu Version + Output để giữ checkpoint. Chạy lại ACTION=\'train\' sẽ tiếp tục "\n                  "job còn thiếu, bỏ qua job hoàn tất. Chỉ đổi ACTION=\'test\' khi đủ 30/30.", flush=True)\n    print("ZIP báo cáo:", _exposure_report_zip(root), flush=True)\n    return str(root)\n\n\ndef exposure_worker_entry():\n    task = json.loads(os.environ["EXPOSURE_WORKER_TASK"])\n    root = Path(task["root"])\n    cfg = _exposure_release_config()\n    _exposure_validate_suite(root, cfg)\n    tf, keras = _exposure_bootstrap_gpu()\n    if task.get("test"):\n        _exposure_evaluate_suite(tf, keras, root, cfg)\n        return\n    from gan_exposure_data import exposure_load_region\n    from gan_exposure_training import exposure_gpu_smoke, exposure_train_job\n    started = time.monotonic()\n    deadline = float(task["runtime"]["session_seconds"])\n    smoke_regions = set()\n    for region, seed, arm in task["jobs"]:\n        remaining = deadline - (time.monotonic() - started)\n        if remaining <= 30:\n            print("Đã tới giới hạn thời gian phiên; giữ checkpoint để tiếp tục.", flush=True)\n            break\n        data = exposure_load_region(root / "data", region, arm)\n        jobdir = root / "jobs" / _exposure_job_id(region, seed, arm)\n        existing = _exposure_inspect_job(jobdir, cfg, data)\n        if existing.get("c_updates", 0) >= task["runtime"]["stop_c_updates"]:\n            print("Đã có đủ bước; bỏ qua:", jobdir.name, existing.get("c_updates"), flush=True)\n            continue\n        smoke_root = root / "gpu_smoke" / f"worker_gpu{os.environ[\'CUDA_VISIBLE_DEVICES\']}_region{region}_{arm}"\n        job_cfg = dict(cfg, gpu_smoke_dir=str(smoke_root))\n        if (region, arm) not in smoke_regions:\n            exposure_gpu_smoke(tf, keras, data, job_cfg, smoke_root)\n            smoke_regions.add((region, arm))\n        remaining = deadline - (time.monotonic() - started)\n        if remaining <= 30:\n            print("Smoke đã qua; gần hết thời gian phiên. Giữ báo cáo để phiên sau tiếp tục.", flush=True)\n            break\n        print("JOB:", jobdir.name, flush=True)\n        exposure_train_job(tf, keras, data, jobdir, job_cfg, seed, region, arm,\n                                    max_c_updates=task["runtime"]["stop_c_updates"],\n                                    save_every=task["runtime"]["save_every"], wall_seconds=remaining)\n        result = _exposure_inspect_job(jobdir, cfg, data)\n        print("JOB STATUS:", jobdir.name, result.get("c_updates"),\n              "completed=", result.get("completed"), flush=True)\n\n\ndef _exposure_metrics(y, prediction, nll):\n    import numpy as np\n    from sklearn.metrics import f1_score, recall_score, cohen_kappa_score\n    y = np.asarray(y, dtype=np.int64)\n    prediction = np.asarray(prediction, dtype=np.int64)\n    labels = np.unique(y)\n    if len(y) == 0:\n        return {"samples": 0, "classes": 0, "OA": None, "AA": None,\n                "Macro_F1": None, "mean_NLL": None, "balanced_NLL": None, "Kappa": None}\n    return {"samples": int(len(y)), "classes": int(len(labels)),\n            "OA": float(np.mean(prediction == y)),\n            "AA": float(recall_score(y, prediction, labels=labels, average="macro", zero_division=0)),\n            "Macro_F1": float(f1_score(y, prediction, labels=labels, average="macro", zero_division=0)),\n            "mean_NLL": float(np.mean(nll)),\n            "balanced_NLL": float(np.mean([np.mean(nll[y == c]) for c in labels])),\n            "Kappa": float(cohen_kappa_score(y, prediction)),\n            "true_class_ids": (labels + 1).tolist(),\n            "prediction_space": "all16; macro/AA averaged only true-present classes"}\n\n\ndef _exposure_evaluate_suite(tf, keras, root, cfg):\n    import numpy as np\n    import pandas as pd\n    from scipy.special import logsumexp, softmax\n    from scipy.stats import t\n    from gan_exposure_data import exposure_load_region, exposure_load_test\n    from gan_exposure_training import exposure_load_classifier\n    _exposure_all_test_gate(root, cfg)\n    out = Path(root) / "test_results"\n    out.mkdir(exist_ok=True)\n    checkpoint_digests = { _exposure_job_id(r,s,a): _exposure_file_hash(\n        Path(root) / "jobs" / _exposure_job_id(r,s,a) / "done.json")\n        for r,s,a in _exposure_expected_jobs() }\n    cache_manifest = out / "manifest.json"\n    if cache_manifest.exists():\n        cached = _exposure_read_json(cache_manifest)\n        if (cached.get("complete") is not True\n                or cached.get("suite_spec_sha256") != _exposure_json_hash(_exposure_suite_spec(cfg))\n                or cached.get("checkpoint_digests") != checkpoint_digests):\n            raise RuntimeError("Test cache không thuộc đúng suite/checkpoints hiện tại.")\n        for name,digest in cached["files_sha256"].items():\n            if Path(name).name != name or _exposure_file_hash(out / name) != digest:\n                raise RuntimeError("Test report cache/hash không khớp.")\n        print("Đã có test report đã kiểm hash; dùng lại, không inference lần nữa.", flush=True)\n        return\n    rows, arrays = [], {}\n    for region in (1, 2, 3):\n        test = exposure_load_test(Path(root) / "data", region)\n        y = np.asarray(test["y_test"], dtype=np.int64)\n        e = np.asarray(test["exposure_counts"], dtype=np.int64)\n        for seed in range(42, 47):\n            for arm in ("blocked", "reuse_control"):\n                jobdir = Path(root) / "jobs" / _exposure_job_id(region, seed, arm)\n                data = exposure_load_region(Path(root) / "data", region, arm)\n                loaded = exposure_load_classifier(tf, keras, jobdir, cfg, data)\n                model, logits_model, order = loaded["C"], loaded["logits"], np.asarray(loaded["order"])\n                if order.shape != (16,) or set(order.tolist()) != set(range(16)):\n                    raise RuntimeError("Classifier class order không đủ 16 lớp.")\n                logits_blocks = []\n                for start in range(0, len(y), 32):\n                    batch = np.asarray(test["X_test"][start:start + 32], dtype=np.float32)\n                    block = np.asarray(logits_model(batch, training=False), dtype=np.float64)\n                    prob = np.asarray(model(batch, training=False), dtype=np.float64)\n                    if block.shape != (len(batch), 16) or not np.isfinite(block).all():\n                        raise RuntimeError("Test logits sai shape hoặc nonfinite.")\n                    np.testing.assert_allclose(softmax(block, axis=1), prob, atol=2e-6, rtol=2e-5)\n                    logits_blocks.append(block)\n                sorted_logits = np.concatenate(logits_blocks)\n                logits = np.empty_like(sorted_logits)\n                logits[:, order] = sorted_logits\n                pred = logits.argmax(axis=1)\n                nll = logsumexp(logits, axis=1) - logits[np.arange(len(y)), y]\n                arrays[(region, seed, arm)] = {"y": y, "pred": pred, "nll": nll, "exposure": e}\n                np.savez_compressed(out / f"{_exposure_job_id(region, seed, arm)}.npz",\n                                    y=y, prediction=pred, logits=logits, nll=nll,\n                                    coords=test["coords_test"], exposure_counts=e)\n                for name, mask in (("all", np.ones(len(y), dtype=bool)), ("exposed", e > 0),\n                                   ("zero_shared", e == 0)):\n                    rows.append({"region": region, "seed": seed, "arm": arm, "subset": name,\n                                 **_exposure_metrics(y[mask], pred[mask], nll[mask])})\n                print("Test:", region, seed, arm, "|", len(y), "mẫu; đủ16outputs.", flush=True)\n                keras.backend.clear_session()\n    conditions = pd.DataFrame(rows)\n    conditions.to_csv(out / "condition_summary.csv", index=False)\n    paired = []\n    for region in (1, 2, 3):\n        for seed in range(42, 47):\n            for subset in ("all", "exposed", "zero_shared"):\n                part = conditions[(conditions.region == region) & (conditions.seed == seed)\n                                  & (conditions.subset == subset)].set_index("arm")\n                row = {"region": region, "seed": seed, "subset": subset,\n                       "samples": int(part.loc["blocked", "samples"])}\n                for metric in ("OA", "AA", "Macro_F1", "mean_NLL", "balanced_NLL"):\n                    row[f"delta_{metric}_control_minus_blocked"] = (\n                        float(part.loc["reuse_control", metric] - part.loc["blocked", metric]))\n                paired.append(row)\n    paired = pd.DataFrame(paired)\n    paired.to_csv(out / "paired_by_region_seed.csv", index=False)\n    pooled = []\n    perclass = []\n    dose_rows = []\n    for seed in range(42, 47):\n        full = {}\n        for arm in ("blocked", "reuse_control"):\n            full[arm] = {key: np.concatenate([arrays[(r, seed, arm)][key] for r in (1, 2, 3)])\n                         for key in ("y", "pred", "nll", "exposure")}\n        for subset in ("all", "exposed", "zero_shared"):\n            e = full["blocked"]["exposure"]\n            mask = np.ones(len(e), bool) if subset == "all" else (e > 0 if subset == "exposed" else e == 0)\n            m = {arm: _exposure_metrics(val["y"][mask], val["pred"][mask], val["nll"][mask])\n                 for arm, val in full.items()}\n            row = {"seed": seed, "subset": subset, "samples": int(mask.sum())}\n            for metric in ("OA", "AA", "Macro_F1", "mean_NLL", "balanced_NLL"):\n                row[f"blocked_{metric}"] = m["blocked"][metric]\n                row[f"control_{metric}"] = m["reuse_control"][metric]\n                row[f"delta_{metric}"] = m["reuse_control"][metric] - m["blocked"][metric]\n            pooled.append(row)\n        for class_id in np.unique(full["blocked"]["y"]):\n            mask = full["blocked"]["y"] == class_id\n            perclass.append({"seed": seed, "class_id": int(class_id + 1), "samples": int(mask.sum()),\n                             "mean_shared_sources": float(full["blocked"]["exposure"][mask].mean()),\n                             "delta_recall": float(np.mean(full["reuse_control"]["pred"][mask] == class_id)\n                                                   - np.mean(full["blocked"]["pred"][mask] == class_id)),\n                             "delta_NLL": float(np.mean(full["reuse_control"]["nll"][mask]\n                                                       - full["blocked"]["nll"][mask]))})\n        for name, low, high in cfg["exposure_bins"]:\n            e = full["blocked"]["exposure"]\n            mask = (e >= low) & (e <= high)\n            if not mask.any():\n                continue\n            b, c = full["blocked"], full["reuse_control"]\n            dose_rows.append({"seed": seed, "bin": name, "samples": int(mask.sum()),\n                              "delta_OA": float(np.mean(c["pred"][mask] == c["y"][mask])\n                                                - np.mean(b["pred"][mask] == b["y"][mask])),\n                              "delta_NLL": float(np.mean(c["nll"][mask] - b["nll"][mask])),\n                              "scope": "descriptive association; not causal dose-response"})\n    pooled = pd.DataFrame(pooled)\n    pooled.to_csv(out / "paired_summary.csv", index=False)\n    pd.DataFrame(perclass).to_csv(out / "per_class_summary.csv", index=False)\n    pd.DataFrame(dose_rows).to_csv(out / "exposure_bins.csv", index=False)\n    region_summary = paired.groupby(["region", "subset"]).agg(\n        mean_delta_OA=("delta_OA_control_minus_blocked", "mean"),\n        sd_delta_OA=("delta_OA_control_minus_blocked", "std"),\n        positive_seeds=("delta_OA_control_minus_blocked", lambda s: int((s > 0).sum())))\n    region_summary.to_csv(out / "region_summary.csv")\n    primary = pooled[pooled.subset == "exposed"].delta_OA.to_numpy(dtype=float)\n    mean, sd = float(primary.mean()), float(primary.std(ddof=1))\n    half = float(t.ppf(.975, len(primary) - 1) * sd / np.sqrt(len(primary)))\n    reg = paired[paired.subset == "exposed"]\n    positive_regions = int((reg.groupby("region").delta_OA_control_minus_blocked.mean() > 0).sum())\n    positive_pairs = int((reg.delta_OA_control_minus_blocked > 0).sum())\n    descriptive_signal = (mean >= cfg["practical_delta_oa"]\n                          and positive_regions >= cfg["minimum_positive_regions"]\n                          and positive_pairs >= cfg["minimum_positive_pairs"])\n    summary = {"primary_samples": 751, "all_samples": 1109, "evaluable_classes": 12,\n               "mean_delta_OA": mean, "sd_across_paired_seed_suites": sd,\n               "conditional_seed_t_interval_95": [mean - half, mean + half],\n               "positive_region_means": positive_regions, "positive_region_seed_pairs": positive_pairs,\n               "registered_descriptive_signal": bool(descriptive_signal),\n               "criterion": {key: cfg[key] for key in ("practical_delta_oa", "minimum_positive_regions",\n                                                        "minimum_positive_pairs")},\n               "interpretation": "Threshold is descriptive, not a statistical or causal proof. "\n                                  "Small-n t CI assumes independent approximately normal paired seed-suite effects, "\n                                  "conditional on fixed3regions. "\n                                  "Not spatial-population uncertainty; not B inflation or isolated generator effect.",\n               "source_intervention": "Different context content/directions remain in contrast.",\n               "training_updates_during_test": 0}\n    if any(int(row["samples"]) != {"all": 1109, "exposed": 751, "zero_shared": 358}[row["subset"]]\n           for row in pooled.to_dict("records")):\n        raise RuntimeError("Pooled test counts khác registered geometry.")\n    _exposure_write_json(out / "summary.json", summary)\n    files = {p.name: _exposure_file_hash(p) for p in out.iterdir()\n             if p.is_file() and p.name != "manifest.json"}\n    _exposure_write_json(out / "manifest.json",\n                         {"complete": True, "suite_spec_sha256": _exposure_json_hash(_exposure_suite_spec(cfg)),\n                          "checkpoint_digests": checkpoint_digests,\n                          "files_sha256": files, "training_updates": 0, "scope": summary["interpretation"]})\n    print("PRIMARY ΔOA:", round(mean * 100, 3), "pp;", positive_regions,\n          "/3 region means positive;", positive_pairs, "/15 pairs positive.", flush=True)\n',
    'gan_exposure_release.py': '# Generated frozen scientific configuration.\nRELEASE_ID = \'4c08e7e92312\'\nCOMPONENT_HASHES = {\'gan_exposure_data.py\': \'bf7cde0d7305f430d8ab85866b7e3b66ea99903cf7d5ff8fac5d6bbe71920733\', \'gan_exposure_training.py\': \'8a1ef7dd13702665cfa56c30ef2653b43076a9c2af67d6a31af17aea50ad5c63\', \'gan_exposure_runner.py\': \'bc5616400f49be3a40d86fa39f69ce019941f2ac9f1886414e2a22a3f465f5be\'}\nSCIENTIFIC_CONFIG = {\'protocol\': \'registered_HyperGAMO_train_source_intervention_v1\', \'target_c_updates\': 8000, \'batch\': 100, \'latent\': 100, \'gen_points\': 7, \'learning_rate\': 0.0002, \'beta1\': 0.5, \'loss\': \'mse\', \'patch_size\': 11, \'pca_components\': 40, \'pca_solver\': \'full\', \'scaler_range\': [-1, 1], \'masked_context_fill\': \'scaled_train_center_mean\', \'image_padding_fill\': 0, \'preprocessing_fit\': \'train_center_spectra_only\', \'checkpoint_selection\': \'fixed8000_Cupdates\', \'seeds\': [42, 43, 44, 45, 46], \'regions\': [[15, 39, 15, 39], [15, 39, 105, 129], [105, 129, 45, 69]], \'arms\': [\'blocked\', \'reuse_control\'], \'test_input\': \'identical_full11x11x40\', \'primary_metric\': \'OA_on_geometry_exposed751_pooled_perseed\', \'practical_delta_oa\': 0.01, \'minimum_positive_regions\': 2, \'minimum_positive_pairs\': 12, \'criterion_scope\': \'registered descriptive rule; not statistical/causal proof\', \'exposure_bins\': [[\'zero\', 0, 0], [\'1_to_20\', 1, 20], [\'21_to_40\', 21, 40], [\'41_to_121\', 41, 121]], \'evaluable_classes\': [2, 3, 4, 5, 6, 8, 10, 11, 12, 13, 14, 15], \'gan_specific_claim\': False, \'historical_B_inflation_claim\': False, \'data_implementation_sha256\': \'bf7cde0d7305f430d8ab85866b7e3b66ea99903cf7d5ff8fac5d6bbe71920733\', \'training_implementation_sha256\': \'8a1ef7dd13702665cfa56c30ef2653b43076a9c2af67d6a31af17aea50ad5c63\', \'region_selection_sha256\': \'c91ea95a6d91df3751b3b13643fb4545234b5e7565960e6b255cef4a887c912c\', \'raw_hsi_sha256\': \'ec2f8808710919d566f70f0d4aa885aae1ddfd42b734aba71c5e12ca65450939\', \'raw_gt_sha256\': \'65c4687a8ab04f6da4789799bc3bc4f6e88bccac3ed6a2e6ae367e5e6b9e429c\'}\nSELECTION_MANIFEST_TEXT = \'{\\r\\n  "GT_sha256": "65c4687a8ab04f6da4789799bc3bc4f6e88bccac3ed6a2e6ae367e5e6b9e429c",\\r\\n  "selection_rule": "Among train-all16/min20 geometry-feasible regions: maximize union evaluable classes across three rectangles, then affected test n, then lexicographic rectangles. No model predictions used.",\\r\\n  "candidate_grid_starts": [\\r\\n    15,\\r\\n    45,\\r\\n    75,\\r\\n    105\\r\\n  ],\\r\\n  "grid": [\\r\\n    {\\r\\n      "rectangle": [\\r\\n        15,\\r\\n        39,\\r\\n        15,\\r\\n        39\\r\\n      ],\\r\\n      "test_n": 437,\\r\\n      "train_n": 9373,\\r\\n      "train_class_counts": [\\r\\n        46,\\r\\n        1047,\\r\\n        685,\\r\\n        155,\\r\\n        473,\\r\\n        724,\\r\\n        28,\\r\\n        478,\\r\\n        20,\\r\\n        936,\\r\\n        2455,\\r\\n        476,\\r\\n        205,\\r\\n        1265,\\r\\n        297,\\r\\n        83\\r\\n      ],\\r\\n      "test_classes": [\\r\\n        2,\\r\\n        3,\\r\\n        4,\\r\\n        10,\\r\\n        12,\\r\\n        15\\r\\n      ],\\r\\n      "train_classes": 16,\\r\\n      "minimum_train_class_n": 20,\\r\\n      "affected_test_n": 285,\\r\\n      "affected_classes": [\\r\\n        2,\\r\\n        3,\\r\\n        4,\\r\\n        10,\\r\\n        12,\\r\\n        15\\r\\n      ],\\r\\n      "control_shared_raw_sources": 596,\\r\\n      "real_train_shared_source_occurrences_per_complete_pass": 15862,\\r\\n      "shared_sources_per_test_patch_histogram": [\\r\\n        152,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        1,\\r\\n        1,\\r\\n        1,\\r\\n        37,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        2,\\r\\n        0,\\r\\n        1,\\r\\n        3,\\r\\n        41,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        1,\\r\\n        1,\\r\\n        1,\\r\\n        0,\\r\\n        6,\\r\\n        1,\\r\\n        43,\\r\\n        0,\\r\\n        0,\\r\\n        1,\\r\\n        1,\\r\\n        1,\\r\\n        0,\\r\\n        3,\\r\\n        3,\\r\\n        1,\\r\\n        1,\\r\\n        34,\\r\\n        2,\\r\\n        0,\\r\\n        1,\\r\\n        1,\\r\\n        3,\\r\\n        0,\\r\\n        4,\\r\\n        0,\\r\\n        2,\\r\\n        3,\\r\\n        39,\\r\\n        0,\\r\\n        3,\\r\\n        5,\\r\\n        0,\\r\\n        0,\\r\\n        5,\\r\\n        0,\\r\\n        2,\\r\\n        0,\\r\\n        4,\\r\\n        0,\\r\\n        5,\\r\\n        1,\\r\\n        2,\\r\\n        0,\\r\\n        0,\\r\\n        2,\\r\\n        5,\\r\\n        0,\\r\\n        2,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        5,\\r\\n        0,\\r\\n        1,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        3,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0\\r\\n      ],\\r\\n      "mean_shared_fraction_all_test": 0.2107343457457874,\\r\\n      "mean_shared_fraction_affected_test": 0.32312599681020737,\\r\\n      "class_mean_shared_fraction": {\\r\\n        "2": 0.2116014345859972,\\r\\n        "3": 0.22941730306517424,\\r\\n        "4": 0.6139315230224321,\\r\\n        "10": 0.45454545454545453,\\r\\n        "12": 0.4027047332832457,\\r\\n        "15": 0.049029622063329927\\r\\n      },\\r\\n      "touching_train_n": 543\\r\\n    },\\r\\n    {\\r\\n      "rectangle": [\\r\\n        15,\\r\\n        39,\\r\\n        45,\\r\\n        69\\r\\n      ],\\r\\n      "test_n": 439,\\r\\n      "train_n": 9515,\\r\\n      "train_class_counts": [\\r\\n        46,\\r\\n        1108,\\r\\n        830,\\r\\n        237,\\r\\n        483,\\r\\n        730,\\r\\n        28,\\r\\n        478,\\r\\n        20,\\r\\n        970,\\r\\n        2455,\\r\\n        274,\\r\\n        205,\\r\\n        1265,\\r\\n        386,\\r\\n        0\\r\\n      ],\\r\\n      "test_classes": [\\r\\n        2,\\r\\n        12,\\r\\n        16\\r\\n      ],\\r\\n      "train_classes": 15,\\r\\n      "minimum_train_class_n": 0,\\r\\n      "affected_test_n": 318,\\r\\n      "affected_classes": [\\r\\n        2,\\r\\n        12,\\r\\n        16\\r\\n      ],\\r\\n      "control_shared_raw_sources": 452,\\r\\n      "real_train_shared_source_occurrences_per_complete_pass": 9557,\\r\\n      "shared_sources_per_test_patch_histogram": [\\r\\n        121,\\r\\n        4,\\r\\n        6,\\r\\n        7,\\r\\n        10,\\r\\n        5,\\r\\n        11,\\r\\n        5,\\r\\n        9,\\r\\n        5,\\r\\n        9,\\r\\n        19,\\r\\n        10,\\r\\n        3,\\r\\n        4,\\r\\n        6,\\r\\n        9,\\r\\n        1,\\r\\n        5,\\r\\n        3,\\r\\n        6,\\r\\n        5,\\r\\n        21,\\r\\n        2,\\r\\n        3,\\r\\n        2,\\r\\n        4,\\r\\n        2,\\r\\n        5,\\r\\n        0,\\r\\n        4,\\r\\n        6,\\r\\n        2,\\r\\n        19,\\r\\n        2,\\r\\n        0,\\r\\n        2,\\r\\n        1,\\r\\n        3,\\r\\n        0,\\r\\n        4,\\r\\n        5,\\r\\n        2,\\r\\n        2,\\r\\n        15,\\r\\n        2,\\r\\n        0,\\r\\n        0,\\r\\n        2,\\r\\n        5,\\r\\n        1,\\r\\n        4,\\r\\n        1,\\r\\n        1,\\r\\n        1,\\r\\n        15,\\r\\n        1,\\r\\n        2,\\r\\n        4,\\r\\n        1,\\r\\n        1,\\r\\n        4,\\r\\n        0,\\r\\n        1,\\r\\n        1,\\r\\n        4,\\r\\n        0,\\r\\n        5,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        1,\\r\\n        2,\\r\\n        4,\\r\\n        0,\\r\\n        1,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        4,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        2,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0\\r\\n      ],\\r\\n      "mean_shared_fraction_all_test": 0.17116286074662548,\\r\\n      "mean_shared_fraction_affected_test": 0.23629086750870626,\\r\\n      "class_mean_shared_fraction": {\\r\\n        "2": 0.23966942148760334,\\r\\n        "12": 0.07770139800726036,\\r\\n        "16": 0.30082644628099175\\r\\n      },\\r\\n      "touching_train_n": 377\\r\\n    },\\r\\n    {\\r\\n      "rectangle": [\\r\\n        15,\\r\\n        39,\\r\\n        75,\\r\\n        99\\r\\n      ],\\r\\n      "test_n": 193,\\r\\n      "train_n": 9775,\\r\\n      "train_class_counts": [\\r\\n        46,\\r\\n        1394,\\r\\n        830,\\r\\n        237,\\r\\n        483,\\r\\n        730,\\r\\n        28,\\r\\n        478,\\r\\n        20,\\r\\n        699,\\r\\n        2394,\\r\\n        593,\\r\\n        205,\\r\\n        1174,\\r\\n        371,\\r\\n        93\\r\\n      ],\\r\\n      "test_classes": [\\r\\n        10,\\r\\n        11,\\r\\n        14\\r\\n      ],\\r\\n      "train_classes": 16,\\r\\n      "minimum_train_class_n": 20,\\r\\n      "affected_test_n": 147,\\r\\n      "affected_classes": [\\r\\n        10,\\r\\n        11,\\r\\n        14\\r\\n      ],\\r\\n      "control_shared_raw_sources": 437,\\r\\n      "real_train_shared_source_occurrences_per_complete_pass": 11726,\\r\\n      "shared_sources_per_test_patch_histogram": [\\r\\n        46,\\r\\n        1,\\r\\n        1,\\r\\n        1,\\r\\n        1,\\r\\n        1,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        9,\\r\\n        1,\\r\\n        1,\\r\\n        1,\\r\\n        1,\\r\\n        1,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        13,\\r\\n        1,\\r\\n        2,\\r\\n        2,\\r\\n        2,\\r\\n        2,\\r\\n        2,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        1,\\r\\n        16,\\r\\n        2,\\r\\n        2,\\r\\n        2,\\r\\n        2,\\r\\n        1,\\r\\n        1,\\r\\n        1,\\r\\n        0,\\r\\n        2,\\r\\n        0,\\r\\n        14,\\r\\n        1,\\r\\n        2,\\r\\n        2,\\r\\n        3,\\r\\n        1,\\r\\n        1,\\r\\n        1,\\r\\n        1,\\r\\n        0,\\r\\n        0,\\r\\n        16,\\r\\n        3,\\r\\n        2,\\r\\n        1,\\r\\n        2,\\r\\n        2,\\r\\n        1,\\r\\n        0,\\r\\n        2,\\r\\n        3,\\r\\n        0,\\r\\n        0,\\r\\n        2,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        3,\\r\\n        0,\\r\\n        1,\\r\\n        0,\\r\\n        1,\\r\\n        0,\\r\\n        0,\\r\\n        1,\\r\\n        1,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        2,\\r\\n        0,\\r\\n        0,\\r\\n        1,\\r\\n        0,\\r\\n        0,\\r\\n        1,\\r\\n        0,\\r\\n        1,\\r\\n        0,\\r\\n        1,\\r\\n        0,\\r\\n        1,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0\\r\\n      ],\\r\\n      "mean_shared_fraction_all_test": 0.2618079047659829,\\r\\n      "mean_shared_fraction_affected_test": 0.343734187890032,\\r\\n      "class_mean_shared_fraction": {\\r\\n        "10": 0.2034416393071437,\\r\\n        "11": 0.7561983471074382,\\r\\n        "14": 0.39729893166700264\\r\\n      },\\r\\n      "touching_train_n": 503\\r\\n    },\\r\\n    {\\r\\n      "rectangle": [\\r\\n        15,\\r\\n        39,\\r\\n        105,\\r\\n        129\\r\\n      ],\\r\\n      "test_n": 330,\\r\\n      "train_n": 9554,\\r\\n      "train_class_counts": [\\r\\n        46,\\r\\n        1404,\\r\\n        830,\\r\\n        237,\\r\\n        483,\\r\\n        730,\\r\\n        28,\\r\\n        332,\\r\\n        20,\\r\\n        960,\\r\\n        2252,\\r\\n        593,\\r\\n        205,\\r\\n        955,\\r\\n        386,\\r\\n        93\\r\\n      ],\\r\\n      "test_classes": [\\r\\n        8,\\r\\n        11,\\r\\n        14\\r\\n      ],\\r\\n      "train_classes": 16,\\r\\n      "minimum_train_class_n": 20,\\r\\n      "affected_test_n": 260,\\r\\n      "affected_classes": [\\r\\n        8,\\r\\n        11,\\r\\n        14\\r\\n      ],\\r\\n      "control_shared_raw_sources": 515,\\r\\n      "real_train_shared_source_occurrences_per_complete_pass": 12193,\\r\\n      "shared_sources_per_test_patch_histogram": [\\r\\n        70,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        1,\\r\\n        1,\\r\\n        1,\\r\\n        1,\\r\\n        30,\\r\\n        0,\\r\\n        1,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        2,\\r\\n        2,\\r\\n        2,\\r\\n        2,\\r\\n        30,\\r\\n        0,\\r\\n        1,\\r\\n        1,\\r\\n        0,\\r\\n        2,\\r\\n        0,\\r\\n        2,\\r\\n        2,\\r\\n        4,\\r\\n        2,\\r\\n        29,\\r\\n        0,\\r\\n        1,\\r\\n        3,\\r\\n        1,\\r\\n        0,\\r\\n        0,\\r\\n        5,\\r\\n        4,\\r\\n        2,\\r\\n        2,\\r\\n        29,\\r\\n        1,\\r\\n        1,\\r\\n        2,\\r\\n        1,\\r\\n        5,\\r\\n        2,\\r\\n        5,\\r\\n        2,\\r\\n        1,\\r\\n        2,\\r\\n        29,\\r\\n        0,\\r\\n        4,\\r\\n        6,\\r\\n        1,\\r\\n        1,\\r\\n        5,\\r\\n        1,\\r\\n        0,\\r\\n        0,\\r\\n        5,\\r\\n        0,\\r\\n        5,\\r\\n        1,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        3,\\r\\n        5,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        6,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        3,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0\\r\\n      ],\\r\\n      "mean_shared_fraction_all_test": 0.25900325569747057,\\r\\n      "mean_shared_fraction_affected_test": 0.32873490146217416,\\r\\n      "class_mean_shared_fraction": {\\r\\n        "8": 0.3961776859504132,\\r\\n        "11": 0.29166666666666663,\\r\\n        "14": 0.21645021645021645\\r\\n      },\\r\\n      "touching_train_n": 433\\r\\n    },\\r\\n    {\\r\\n      "rectangle": [\\r\\n        45,\\r\\n        69,\\r\\n        15,\\r\\n        39\\r\\n      ],\\r\\n      "test_n": 501,\\r\\n      "train_n": 9331,\\r\\n      "train_class_counts": [\\r\\n        46,\\r\\n        1314,\\r\\n        698,\\r\\n        118,\\r\\n        457,\\r\\n        460,\\r\\n        28,\\r\\n        478,\\r\\n        0,\\r\\n        972,\\r\\n        2316,\\r\\n        495,\\r\\n        205,\\r\\n        1265,\\r\\n        386,\\r\\n        93\\r\\n      ],\\r\\n      "test_classes": [\\r\\n        2,\\r\\n        3,\\r\\n        4,\\r\\n        6,\\r\\n        9,\\r\\n        11,\\r\\n        12\\r\\n      ],\\r\\n      "train_classes": 15,\\r\\n      "minimum_train_class_n": 0,\\r\\n      "affected_test_n": 343,\\r\\n      "affected_classes": [\\r\\n        2,\\r\\n        3,\\r\\n        4,\\r\\n        6,\\r\\n        9,\\r\\n        11,\\r\\n        12\\r\\n      ],\\r\\n      "control_shared_raw_sources": 600,\\r\\n      "real_train_shared_source_occurrences_per_complete_pass": 16718,\\r\\n      "shared_sources_per_test_patch_histogram": [\\r\\n        158,\\r\\n        4,\\r\\n        3,\\r\\n        1,\\r\\n        2,\\r\\n        1,\\r\\n        2,\\r\\n        0,\\r\\n        1,\\r\\n        1,\\r\\n        0,\\r\\n        42,\\r\\n        3,\\r\\n        1,\\r\\n        0,\\r\\n        2,\\r\\n        1,\\r\\n        1,\\r\\n        0,\\r\\n        1,\\r\\n        1,\\r\\n        3,\\r\\n        31,\\r\\n        2,\\r\\n        1,\\r\\n        0,\\r\\n        2,\\r\\n        0,\\r\\n        1,\\r\\n        0,\\r\\n        2,\\r\\n        6,\\r\\n        0,\\r\\n        41,\\r\\n        3,\\r\\n        1,\\r\\n        0,\\r\\n        2,\\r\\n        1,\\r\\n        1,\\r\\n        3,\\r\\n        7,\\r\\n        1,\\r\\n        0,\\r\\n        42,\\r\\n        3,\\r\\n        1,\\r\\n        0,\\r\\n        2,\\r\\n        6,\\r\\n        2,\\r\\n        6,\\r\\n        2,\\r\\n        1,\\r\\n        0,\\r\\n        42,\\r\\n        3,\\r\\n        4,\\r\\n        5,\\r\\n        2,\\r\\n        3,\\r\\n        7,\\r\\n        1,\\r\\n        0,\\r\\n        1,\\r\\n        6,\\r\\n        1,\\r\\n        5,\\r\\n        1,\\r\\n        1,\\r\\n        0,\\r\\n        0,\\r\\n        4,\\r\\n        6,\\r\\n        0,\\r\\n        0,\\r\\n        1,\\r\\n        0,\\r\\n        1,\\r\\n        6,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        3,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0\\r\\n      ],\\r\\n      "mean_shared_fraction_all_test": 0.21881856122465812,\\r\\n      "mean_shared_fraction_affected_test": 0.31961544948557935,\\r\\n      "class_mean_shared_fraction": {\\r\\n        "2": 0.48760330578512395,\\r\\n        "3": 0.3230653643876784,\\r\\n        "4": 0.3829201101928374,\\r\\n        "6": 0.12545725511448313,\\r\\n        "9": 0.1515151515151515,\\r\\n        "11": 0.4079639368895567,\\r\\n        "12": 0.15151515151515155\\r\\n      },\\r\\n      "touching_train_n": 619\\r\\n    },\\r\\n    {\\r\\n      "rectangle": [\\r\\n        45,\\r\\n        69,\\r\\n        45,\\r\\n        69\\r\\n      ],\\r\\n      "test_n": 392,\\r\\n      "train_n": 9465,\\r\\n      "train_class_counts": [\\r\\n        46,\\r\\n        1146,\\r\\n        830,\\r\\n        237,\\r\\n        483,\\r\\n        730,\\r\\n        28,\\r\\n        478,\\r\\n        20,\\r\\n        926,\\r\\n        1999,\\r\\n        593,\\r\\n        205,\\r\\n        1265,\\r\\n        386,\\r\\n        93\\r\\n      ],\\r\\n      "test_classes": [\\r\\n        2,\\r\\n        11\\r\\n      ],\\r\\n      "train_classes": 16,\\r\\n      "minimum_train_class_n": 20,\\r\\n      "affected_test_n": 284,\\r\\n      "affected_classes": [\\r\\n        2,\\r\\n        11\\r\\n      ],\\r\\n      "control_shared_raw_sources": 572,\\r\\n      "real_train_shared_source_occurrences_per_complete_pass": 15701,\\r\\n      "shared_sources_per_test_patch_histogram": [\\r\\n        108,\\r\\n        0,\\r\\n        0,\\r\\n        1,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        3,\\r\\n        1,\\r\\n        1,\\r\\n        31,\\r\\n        0,\\r\\n        0,\\r\\n        1,\\r\\n        0,\\r\\n        3,\\r\\n        0,\\r\\n        2,\\r\\n        0,\\r\\n        1,\\r\\n        2,\\r\\n        35,\\r\\n        0,\\r\\n        3,\\r\\n        2,\\r\\n        0,\\r\\n        1,\\r\\n        1,\\r\\n        0,\\r\\n        2,\\r\\n        4,\\r\\n        4,\\r\\n        33,\\r\\n        1,\\r\\n        0,\\r\\n        2,\\r\\n        1,\\r\\n        1,\\r\\n        1,\\r\\n        7,\\r\\n        4,\\r\\n        0,\\r\\n        1,\\r\\n        35,\\r\\n        1,\\r\\n        2,\\r\\n        1,\\r\\n        3,\\r\\n        3,\\r\\n        1,\\r\\n        4,\\r\\n        4,\\r\\n        1,\\r\\n        0,\\r\\n        34,\\r\\n        1,\\r\\n        3,\\r\\n        7,\\r\\n        1,\\r\\n        1,\\r\\n        5,\\r\\n        0,\\r\\n        2,\\r\\n        2,\\r\\n        3,\\r\\n        0,\\r\\n        5,\\r\\n        2,\\r\\n        0,\\r\\n        1,\\r\\n        0,\\r\\n        2,\\r\\n        4,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        1,\\r\\n        4,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        2,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0\\r\\n      ],\\r\\n      "mean_shared_fraction_all_test": 0.23294400404790017,\\r\\n      "mean_shared_fraction_affected_test": 0.3215283436154115,\\r\\n      "class_mean_shared_fraction": {\\r\\n        "2": 0.3484204835096934,\\r\\n        "11": 0.1895896766710164\\r\\n      },\\r\\n      "touching_train_n": 601\\r\\n    },\\r\\n    {\\r\\n      "rectangle": [\\r\\n        45,\\r\\n        69,\\r\\n        75,\\r\\n        99\\r\\n      ],\\r\\n      "test_n": 547,\\r\\n      "train_n": 9334,\\r\\n      "train_class_counts": [\\r\\n        0,\\r\\n        1176,\\r\\n        830,\\r\\n        237,\\r\\n        483,\\r\\n        730,\\r\\n        28,\\r\\n        478,\\r\\n        20,\\r\\n        399,\\r\\n        2411,\\r\\n        593,\\r\\n        205,\\r\\n        1265,\\r\\n        386,\\r\\n        93\\r\\n      ],\\r\\n      "test_classes": [\\r\\n        1,\\r\\n        2,\\r\\n        10,\\r\\n        11\\r\\n      ],\\r\\n      "train_classes": 15,\\r\\n      "minimum_train_class_n": 0,\\r\\n      "affected_test_n": 337,\\r\\n      "affected_classes": [\\r\\n        1,\\r\\n        2,\\r\\n        10,\\r\\n        11\\r\\n      ],\\r\\n      "control_shared_raw_sources": 578,\\r\\n      "real_train_shared_source_occurrences_per_complete_pass": 14548,\\r\\n      "shared_sources_per_test_patch_histogram": [\\r\\n        210,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        11,\\r\\n        45,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        1,\\r\\n        12,\\r\\n        44,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        1,\\r\\n        0,\\r\\n        2,\\r\\n        2,\\r\\n        11,\\r\\n        40,\\r\\n        1,\\r\\n        0,\\r\\n        0,\\r\\n        1,\\r\\n        0,\\r\\n        1,\\r\\n        4,\\r\\n        3,\\r\\n        0,\\r\\n        11,\\r\\n        38,\\r\\n        1,\\r\\n        1,\\r\\n        0,\\r\\n        2,\\r\\n        4,\\r\\n        3,\\r\\n        2,\\r\\n        1,\\r\\n        2,\\r\\n        10,\\r\\n        36,\\r\\n        2,\\r\\n        4,\\r\\n        3,\\r\\n        1,\\r\\n        2,\\r\\n        3,\\r\\n        0,\\r\\n        1,\\r\\n        2,\\r\\n        5,\\r\\n        2,\\r\\n        4,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        1,\\r\\n        4,\\r\\n        4,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        2,\\r\\n        4,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        1,\\r\\n        2,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0\\r\\n      ],\\r\\n      "mean_shared_fraction_all_test": 0.1916841675857797,\\r\\n      "mean_shared_fraction_affected_test": 0.31113127498344656,\\r\\n      "class_mean_shared_fraction": {\\r\\n        "1": 0.3921028466483012,\\r\\n        "2": 0.3255280073461891,\\r\\n        "10": 0.14354169091685096,\\r\\n        "11": 0.565162110616656\\r\\n      },\\r\\n      "touching_train_n": 511\\r\\n    },\\r\\n    {\\r\\n      "rectangle": [\\r\\n        45,\\r\\n        69,\\r\\n        105,\\r\\n        129\\r\\n      ],\\r\\n      "test_n": 331,\\r\\n      "train_n": 9587,\\r\\n      "train_class_counts": [\\r\\n        43,\\r\\n        1282,\\r\\n        830,\\r\\n        237,\\r\\n        451,\\r\\n        730,\\r\\n        16,\\r\\n        198,\\r\\n        20,\\r\\n        906,\\r\\n        2332,\\r\\n        593,\\r\\n        205,\\r\\n        1265,\\r\\n        386,\\r\\n        93\\r\\n      ],\\r\\n      "test_classes": [\\r\\n        2,\\r\\n        5,\\r\\n        8,\\r\\n        10,\\r\\n        11\\r\\n      ],\\r\\n      "train_classes": 16,\\r\\n      "minimum_train_class_n": 16,\\r\\n      "affected_test_n": 217,\\r\\n      "affected_classes": [\\r\\n        2,\\r\\n        5,\\r\\n        8,\\r\\n        10,\\r\\n        11\\r\\n      ],\\r\\n      "control_shared_raw_sources": 469,\\r\\n      "real_train_shared_source_occurrences_per_complete_pass": 12745,\\r\\n      "shared_sources_per_test_patch_histogram": [\\r\\n        114,\\r\\n        0,\\r\\n        2,\\r\\n        1,\\r\\n        2,\\r\\n        1,\\r\\n        1,\\r\\n        1,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        26,\\r\\n        0,\\r\\n        2,\\r\\n        1,\\r\\n        2,\\r\\n        1,\\r\\n        1,\\r\\n        1,\\r\\n        0,\\r\\n        0,\\r\\n        2,\\r\\n        26,\\r\\n        0,\\r\\n        2,\\r\\n        1,\\r\\n        2,\\r\\n        1,\\r\\n        1,\\r\\n        1,\\r\\n        0,\\r\\n        4,\\r\\n        0,\\r\\n        19,\\r\\n        0,\\r\\n        1,\\r\\n        1,\\r\\n        1,\\r\\n        1,\\r\\n        1,\\r\\n        3,\\r\\n        3,\\r\\n        0,\\r\\n        0,\\r\\n        30,\\r\\n        0,\\r\\n        2,\\r\\n        1,\\r\\n        2,\\r\\n        4,\\r\\n        2,\\r\\n        5,\\r\\n        1,\\r\\n        0,\\r\\n        1,\\r\\n        29,\\r\\n        0,\\r\\n        2,\\r\\n        4,\\r\\n        1,\\r\\n        0,\\r\\n        4,\\r\\n        0,\\r\\n        1,\\r\\n        0,\\r\\n        4,\\r\\n        0,\\r\\n        3,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        2,\\r\\n        3,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        3,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        1,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0\\r\\n      ],\\r\\n      "mean_shared_fraction_all_test": 0.20241691842900303,\\r\\n      "mean_shared_fraction_affected_test": 0.3087557603686636,\\r\\n      "class_mean_shared_fraction": {\\r\\n        "2": 0.4483471074380166,\\r\\n        "5": 0.3636363636363636,\\r\\n        "8": 0.22865013774104684,\\r\\n        "10": 0.14244044725328145,\\r\\n        "11": 0.07064782724606772\\r\\n      },\\r\\n      "touching_train_n": 466\\r\\n    },\\r\\n    {\\r\\n      "rectangle": [\\r\\n        75,\\r\\n        99,\\r\\n        15,\\r\\n        39\\r\\n      ],\\r\\n      "test_n": 361,\\r\\n      "train_n": 9500,\\r\\n      "train_class_counts": [\\r\\n        46,\\r\\n        1420,\\r\\n        819,\\r\\n        237,\\r\\n        300,\\r\\n        710,\\r\\n        28,\\r\\n        478,\\r\\n        18,\\r\\n        972,\\r\\n        1930,\\r\\n        593,\\r\\n        205,\\r\\n        1265,\\r\\n        386,\\r\\n        93\\r\\n      ],\\r\\n      "test_classes": [\\r\\n        5,\\r\\n        11\\r\\n      ],\\r\\n      "train_classes": 16,\\r\\n      "minimum_train_class_n": 18,\\r\\n      "affected_test_n": 277,\\r\\n      "affected_classes": [\\r\\n        5,\\r\\n        11\\r\\n      ],\\r\\n      "control_shared_raw_sources": 515,\\r\\n      "real_train_shared_source_occurrences_per_complete_pass": 14769,\\r\\n      "shared_sources_per_test_patch_histogram": [\\r\\n        84,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        39,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        3,\\r\\n        40,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        6,\\r\\n        0,\\r\\n        41,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        3,\\r\\n        6,\\r\\n        0,\\r\\n        0,\\r\\n        41,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        6,\\r\\n        0,\\r\\n        6,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        41,\\r\\n        0,\\r\\n        3,\\r\\n        6,\\r\\n        0,\\r\\n        0,\\r\\n        6,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        6,\\r\\n        0,\\r\\n        6,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        3,\\r\\n        6,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        6,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        3,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0\\r\\n      ],\\r\\n      "mean_shared_fraction_all_test": 0.25173416359515577,\\r\\n      "mean_shared_fraction_affected_test": 0.3280723215084882,\\r\\n      "class_mean_shared_fraction": {\\r\\n        "5": 0.2871002515271289,\\r\\n        "11": 0.23963869857752926\\r\\n      },\\r\\n      "touching_train_n": 522\\r\\n    },\\r\\n    {\\r\\n      "rectangle": [\\r\\n        75,\\r\\n        99,\\r\\n        45,\\r\\n        69\\r\\n      ],\\r\\n      "test_n": 543,\\r\\n      "train_n": 9253,\\r\\n      "train_class_counts": [\\r\\n        46,\\r\\n        1381,\\r\\n        830,\\r\\n        237,\\r\\n        483,\\r\\n        612,\\r\\n        28,\\r\\n        478,\\r\\n        20,\\r\\n        972,\\r\\n        1624,\\r\\n        593,\\r\\n        205,\\r\\n        1265,\\r\\n        386,\\r\\n        93\\r\\n      ],\\r\\n      "test_classes": [\\r\\n        6,\\r\\n        11\\r\\n      ],\\r\\n      "train_classes": 16,\\r\\n      "minimum_train_class_n": 20,\\r\\n      "affected_test_n": 318,\\r\\n      "affected_classes": [\\r\\n        6,\\r\\n        11\\r\\n      ],\\r\\n      "control_shared_raw_sources": 600,\\r\\n      "real_train_shared_source_occurrences_per_complete_pass": 20372,\\r\\n      "shared_sources_per_test_patch_histogram": [\\r\\n        225,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        60,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        4,\\r\\n        51,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        7,\\r\\n        0,\\r\\n        29,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        3,\\r\\n        4,\\r\\n        0,\\r\\n        0,\\r\\n        52,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        3,\\r\\n        0,\\r\\n        7,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        52,\\r\\n        0,\\r\\n        1,\\r\\n        6,\\r\\n        0,\\r\\n        0,\\r\\n        7,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        5,\\r\\n        0,\\r\\n        6,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        4,\\r\\n        5,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        8,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        4,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0\\r\\n      ],\\r\\n      "mean_shared_fraction_all_test": 0.18507526292558937,\\r\\n      "mean_shared_fraction_affected_test": 0.31602474141067616,\\r\\n      "class_mean_shared_fraction": {\\r\\n        "6": 0.47107438016528924,\\r\\n        "11": 0.17069232859632014\\r\\n      },\\r\\n      "touching_train_n": 709\\r\\n    },\\r\\n    {\\r\\n      "rectangle": [\\r\\n        75,\\r\\n        99,\\r\\n        75,\\r\\n        99\\r\\n      ],\\r\\n      "test_n": 271,\\r\\n      "train_n": 9742,\\r\\n      "train_class_counts": [\\r\\n        35,\\r\\n        1132,\\r\\n        830,\\r\\n        237,\\r\\n        483,\\r\\n        574,\\r\\n        28,\\r\\n        478,\\r\\n        20,\\r\\n        966,\\r\\n        2417,\\r\\n        593,\\r\\n        205,\\r\\n        1265,\\r\\n        386,\\r\\n        93\\r\\n      ],\\r\\n      "test_classes": [\\r\\n        2,\\r\\n        6\\r\\n      ],\\r\\n      "train_classes": 16,\\r\\n      "minimum_train_class_n": 20,\\r\\n      "affected_test_n": 220,\\r\\n      "affected_classes": [\\r\\n        2,\\r\\n        6\\r\\n      ],\\r\\n      "control_shared_raw_sources": 439,\\r\\n      "real_train_shared_source_occurrences_per_complete_pass": 10601,\\r\\n      "shared_sources_per_test_patch_histogram": [\\r\\n        51,\\r\\n        0,\\r\\n        0,\\r\\n        1,\\r\\n        1,\\r\\n        0,\\r\\n        0,\\r\\n        1,\\r\\n        0,\\r\\n        1,\\r\\n        0,\\r\\n        26,\\r\\n        0,\\r\\n        0,\\r\\n        1,\\r\\n        2,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        2,\\r\\n        1,\\r\\n        2,\\r\\n        25,\\r\\n        0,\\r\\n        1,\\r\\n        1,\\r\\n        1,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        1,\\r\\n        5,\\r\\n        0,\\r\\n        25,\\r\\n        0,\\r\\n        1,\\r\\n        1,\\r\\n        1,\\r\\n        0,\\r\\n        0,\\r\\n        2,\\r\\n        5,\\r\\n        1,\\r\\n        0,\\r\\n        25,\\r\\n        0,\\r\\n        1,\\r\\n        1,\\r\\n        1,\\r\\n        4,\\r\\n        0,\\r\\n        4,\\r\\n        1,\\r\\n        1,\\r\\n        0,\\r\\n        35,\\r\\n        0,\\r\\n        3,\\r\\n        5,\\r\\n        1,\\r\\n        0,\\r\\n        4,\\r\\n        0,\\r\\n        1,\\r\\n        1,\\r\\n        4,\\r\\n        0,\\r\\n        4,\\r\\n        1,\\r\\n        1,\\r\\n        0,\\r\\n        0,\\r\\n        2,\\r\\n        4,\\r\\n        1,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        5,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        2,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0\\r\\n      ],\\r\\n      "mean_shared_fraction_all_test": 0.26912872434509466,\\r\\n      "mean_shared_fraction_affected_test": 0.3315176558978211,\\r\\n      "class_mean_shared_fraction": {\\r\\n        "2": 0.23765005048427507,\\r\\n        "6": 0.4082644628099173\\r\\n      },\\r\\n      "touching_train_n": 383\\r\\n    },\\r\\n    {\\r\\n      "rectangle": [\\r\\n        75,\\r\\n        99,\\r\\n        105,\\r\\n        129\\r\\n      ],\\r\\n      "test_n": 43,\\r\\n      "train_n": 10174,\\r\\n      "train_class_counts": [\\r\\n        46,\\r\\n        1428,\\r\\n        830,\\r\\n        237,\\r\\n        436,\\r\\n        730,\\r\\n        0,\\r\\n        478,\\r\\n        20,\\r\\n        972,\\r\\n        2455,\\r\\n        593,\\r\\n        205,\\r\\n        1265,\\r\\n        386,\\r\\n        93\\r\\n      ],\\r\\n      "test_classes": [\\r\\n        5,\\r\\n        7\\r\\n      ],\\r\\n      "train_classes": 15,\\r\\n      "minimum_train_class_n": 0,\\r\\n      "affected_test_n": 36,\\r\\n      "affected_classes": [\\r\\n        5,\\r\\n        7\\r\\n      ],\\r\\n      "control_shared_raw_sources": 110,\\r\\n      "real_train_shared_source_occurrences_per_complete_pass": 983,\\r\\n      "shared_sources_per_test_patch_histogram": [\\r\\n        7,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        4,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        4,\\r\\n        0,\\r\\n        0,\\r\\n        1,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        1,\\r\\n        0,\\r\\n        0,\\r\\n        4,\\r\\n        0,\\r\\n        2,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        2,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        4,\\r\\n        2,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        2,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        6,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        2,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        1,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        1,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0\\r\\n      ],\\r\\n      "mean_shared_fraction_all_test": 0.27291946953680574,\\r\\n      "mean_shared_fraction_affected_test": 0.32598714416896246,\\r\\n      "class_mean_shared_fraction": {\\r\\n        "5": 0.20202020202020204,\\r\\n        "7": 0.3925619834710744\\r\\n      },\\r\\n      "touching_train_n": 40\\r\\n    },\\r\\n    {\\r\\n      "rectangle": [\\r\\n        105,\\r\\n        129,\\r\\n        15,\\r\\n        39\\r\\n      ],\\r\\n      "test_n": 391,\\r\\n      "train_n": 9508,\\r\\n      "train_class_counts": [\\r\\n        46,\\r\\n        1428,\\r\\n        578,\\r\\n        237,\\r\\n        483,\\r\\n        730,\\r\\n        28,\\r\\n        478,\\r\\n        20,\\r\\n        972,\\r\\n        2155,\\r\\n        593,\\r\\n        16,\\r\\n        1265,\\r\\n        386,\\r\\n        93\\r\\n      ],\\r\\n      "test_classes": [\\r\\n        3,\\r\\n        11,\\r\\n        13\\r\\n      ],\\r\\n      "train_classes": 16,\\r\\n      "minimum_train_class_n": 16,\\r\\n      "affected_test_n": 211,\\r\\n      "affected_classes": [\\r\\n        3,\\r\\n        11,\\r\\n        13\\r\\n      ],\\r\\n      "control_shared_raw_sources": 326,\\r\\n      "real_train_shared_source_occurrences_per_complete_pass": 8052,\\r\\n      "shared_sources_per_test_patch_histogram": [\\r\\n        180,\\r\\n        1,\\r\\n        2,\\r\\n        2,\\r\\n        2,\\r\\n        1,\\r\\n        3,\\r\\n        1,\\r\\n        2,\\r\\n        2,\\r\\n        2,\\r\\n        31,\\r\\n        2,\\r\\n        0,\\r\\n        1,\\r\\n        1,\\r\\n        1,\\r\\n        0,\\r\\n        2,\\r\\n        0,\\r\\n        1,\\r\\n        3,\\r\\n        31,\\r\\n        0,\\r\\n        1,\\r\\n        0,\\r\\n        0,\\r\\n        1,\\r\\n        0,\\r\\n        0,\\r\\n        1,\\r\\n        4,\\r\\n        0,\\r\\n        31,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        2,\\r\\n        4,\\r\\n        0,\\r\\n        0,\\r\\n        24,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        4,\\r\\n        0,\\r\\n        3,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        24,\\r\\n        0,\\r\\n        2,\\r\\n        3,\\r\\n        0,\\r\\n        0,\\r\\n        3,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        3,\\r\\n        0,\\r\\n        3,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        1,\\r\\n        3,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        2,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        1,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0\\r\\n      ],\\r\\n      "mean_shared_fraction_all_test": 0.14609710215383312,\\r\\n      "mean_shared_fraction_affected_test": 0.27072970114762446,\\r\\n      "class_mean_shared_fraction": {\\r\\n        "3": 0.15199425080848006,\\r\\n        "11": 0.21487603305785125,\\r\\n        "13": 0.07321537522879804\\r\\n      },\\r\\n      "touching_train_n": 293\\r\\n    },\\r\\n    {\\r\\n      "rectangle": [\\r\\n        105,\\r\\n        129,\\r\\n        45,\\r\\n        69\\r\\n      ],\\r\\n      "test_n": 342,\\r\\n      "train_n": 9582,\\r\\n      "train_class_counts": [\\r\\n        46,\\r\\n        1428,\\r\\n        767,\\r\\n        237,\\r\\n        395,\\r\\n        495,\\r\\n        28,\\r\\n        478,\\r\\n        20,\\r\\n        972,\\r\\n        2230,\\r\\n        593,\\r\\n        149,\\r\\n        1265,\\r\\n        386,\\r\\n        93\\r\\n      ],\\r\\n      "test_classes": [\\r\\n        3,\\r\\n        5,\\r\\n        6,\\r\\n        11,\\r\\n        13\\r\\n      ],\\r\\n      "train_classes": 16,\\r\\n      "minimum_train_class_n": 20,\\r\\n      "affected_test_n": 206,\\r\\n      "affected_classes": [\\r\\n        3,\\r\\n        5,\\r\\n        6,\\r\\n        11,\\r\\n        13\\r\\n      ],\\r\\n      "control_shared_raw_sources": 396,\\r\\n      "real_train_shared_source_occurrences_per_complete_pass": 10911,\\r\\n      "shared_sources_per_test_patch_histogram": [\\r\\n        136,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        1,\\r\\n        1,\\r\\n        1,\\r\\n        1,\\r\\n        1,\\r\\n        30,\\r\\n        1,\\r\\n        0,\\r\\n        1,\\r\\n        0,\\r\\n        1,\\r\\n        0,\\r\\n        2,\\r\\n        0,\\r\\n        1,\\r\\n        3,\\r\\n        30,\\r\\n        0,\\r\\n        2,\\r\\n        0,\\r\\n        0,\\r\\n        1,\\r\\n        1,\\r\\n        0,\\r\\n        2,\\r\\n        4,\\r\\n        1,\\r\\n        14,\\r\\n        0,\\r\\n        1,\\r\\n        1,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        4,\\r\\n        3,\\r\\n        0,\\r\\n        0,\\r\\n        28,\\r\\n        1,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        3,\\r\\n        1,\\r\\n        5,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        28,\\r\\n        0,\\r\\n        1,\\r\\n        5,\\r\\n        0,\\r\\n        0,\\r\\n        5,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        4,\\r\\n        0,\\r\\n        5,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        2,\\r\\n        4,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        4,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        2,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0\\r\\n      ],\\r\\n      "mean_shared_fraction_all_test": 0.18761780484268523,\\r\\n      "mean_shared_fraction_affected_test": 0.31148198668057453,\\r\\n      "class_mean_shared_fraction": {\\r\\n        "3": 0.4896694214876033,\\r\\n        "5": 0.0340909090909091,\\r\\n        "6": 0.2126509853782581,\\r\\n        "11": 0.23057851239669422,\\r\\n        "13": 0.40909090909090906\\r\\n      },\\r\\n      "touching_train_n": 414\\r\\n    },\\r\\n    {\\r\\n      "rectangle": [\\r\\n        105,\\r\\n        129,\\r\\n        75,\\r\\n        99\\r\\n      ],\\r\\n      "test_n": 258,\\r\\n      "train_n": 9699,\\r\\n      "train_class_counts": [\\r\\n        46,\\r\\n        1428,\\r\\n        830,\\r\\n        237,\\r\\n        483,\\r\\n        553,\\r\\n        28,\\r\\n        478,\\r\\n        20,\\r\\n        972,\\r\\n        2455,\\r\\n        593,\\r\\n        205,\\r\\n        892,\\r\\n        386,\\r\\n        93\\r\\n      ],\\r\\n      "test_classes": [\\r\\n        6,\\r\\n        14\\r\\n      ],\\r\\n      "train_classes": 16,\\r\\n      "minimum_train_class_n": 20,\\r\\n      "affected_test_n": 163,\\r\\n      "affected_classes": [\\r\\n        6,\\r\\n        14\\r\\n      ],\\r\\n      "control_shared_raw_sources": 415,\\r\\n      "real_train_shared_source_occurrences_per_complete_pass": 10509,\\r\\n      "shared_sources_per_test_patch_histogram": [\\r\\n        95,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        24,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        1,\\r\\n        24,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        2,\\r\\n        0,\\r\\n        24,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        1,\\r\\n        2,\\r\\n        0,\\r\\n        0,\\r\\n        25,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        2,\\r\\n        0,\\r\\n        3,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        26,\\r\\n        0,\\r\\n        1,\\r\\n        4,\\r\\n        0,\\r\\n        0,\\r\\n        4,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        4,\\r\\n        0,\\r\\n        4,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        2,\\r\\n        4,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        4,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        2,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0\\r\\n      ],\\r\\n      "mean_shared_fraction_all_test": 0.20991094881158306,\\r\\n      "mean_shared_fraction_affected_test": 0.33225168584900877,\\r\\n      "class_mean_shared_fraction": {\\r\\n        "6": 0.2839840410373326,\\r\\n        "14": 0.1884297520661157\\r\\n      },\\r\\n      "touching_train_n": 351\\r\\n    },\\r\\n    {\\r\\n      "rectangle": [\\r\\n        105,\\r\\n        129,\\r\\n        105,\\r\\n        129\\r\\n      ],\\r\\n      "test_n": 271,\\r\\n      "train_n": 9788,\\r\\n      "train_class_counts": [\\r\\n        46,\\r\\n        1428,\\r\\n        830,\\r\\n        237,\\r\\n        483,\\r\\n        730,\\r\\n        28,\\r\\n        478,\\r\\n        20,\\r\\n        972,\\r\\n        2455,\\r\\n        593,\\r\\n        205,\\r\\n        804,\\r\\n        386,\\r\\n        93\\r\\n      ],\\r\\n      "test_classes": [\\r\\n        14\\r\\n      ],\\r\\n      "train_classes": 16,\\r\\n      "minimum_train_class_n": 20,\\r\\n      "affected_test_n": 150,\\r\\n      "affected_classes": [\\r\\n        14\\r\\n      ],\\r\\n      "control_shared_raw_sources": 230,\\r\\n      "real_train_shared_source_occurrences_per_complete_pass": 6287,\\r\\n      "shared_sources_per_test_patch_histogram": [\\r\\n        121,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        1,\\r\\n        1,\\r\\n        1,\\r\\n        1,\\r\\n        21,\\r\\n        0,\\r\\n        0,\\r\\n        1,\\r\\n        0,\\r\\n        1,\\r\\n        0,\\r\\n        1,\\r\\n        0,\\r\\n        1,\\r\\n        2,\\r\\n        21,\\r\\n        0,\\r\\n        1,\\r\\n        0,\\r\\n        0,\\r\\n        1,\\r\\n        1,\\r\\n        0,\\r\\n        1,\\r\\n        2,\\r\\n        1,\\r\\n        21,\\r\\n        0,\\r\\n        1,\\r\\n        1,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        3,\\r\\n        2,\\r\\n        0,\\r\\n        0,\\r\\n        21,\\r\\n        1,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        2,\\r\\n        1,\\r\\n        2,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        21,\\r\\n        0,\\r\\n        1,\\r\\n        2,\\r\\n        0,\\r\\n        0,\\r\\n        2,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        2,\\r\\n        0,\\r\\n        2,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        1,\\r\\n        2,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        2,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        1,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0,\\r\\n        0\\r\\n      ],\\r\\n      "mean_shared_fraction_all_test": 0.16467933274374066,\\r\\n      "mean_shared_fraction_affected_test": 0.2975206611570248,\\r\\n      "class_mean_shared_fraction": {\\r\\n        "14": 0.16467933274374066\\r\\n      },\\r\\n      "touching_train_n": 202\\r\\n    }\\r\\n  ],\\r\\n  "feasible_triplet_count": 84,\\r\\n  "selected": {\\r\\n    "union_classes": [\\r\\n      2,\\r\\n      3,\\r\\n      4,\\r\\n      5,\\r\\n      6,\\r\\n      8,\\r\\n      10,\\r\\n      11,\\r\\n      12,\\r\\n      13,\\r\\n      14,\\r\\n      15\\r\\n    ],\\r\\n    "class_count": 12,\\r\\n    "affected_n": 751,\\r\\n    "rectangles": [\\r\\n      [\\r\\n        15,\\r\\n        39,\\r\\n        15,\\r\\n        39\\r\\n      ],\\r\\n      [\\r\\n        15,\\r\\n        39,\\r\\n        105,\\r\\n        129\\r\\n      ],\\r\\n      [\\r\\n        105,\\r\\n        129,\\r\\n        45,\\r\\n        69\\r\\n      ]\\r\\n    ]\\r\\n  },\\r\\n  "model_training_performed": false\\r\\n}\\r\\n\'\n',
}

from pathlib import Path as _ExposurePath
import hashlib as _exposure_hashlib
import sys as _exposure_sys

_exposure_release_id = '4c08e7e92312'
_exposure_code_root = _ExposurePath(OUTPUT_ROOT) / ("_gan_exposure_code_" + _exposure_release_id)
_exposure_code_root.mkdir(parents=True, exist_ok=True)
for _exposure_name, _exposure_source in EXPOSURE_COMPONENTS.items():
    _exposure_path = _exposure_code_root / _exposure_name
    _exposure_bytes = _exposure_source.encode("utf-8")
    if _exposure_path.exists() and _exposure_path.read_bytes() != _exposure_bytes:
        raise RuntimeError("Runner source khác release: " + str(_exposure_path))
    if not _exposure_path.exists():
        _exposure_path.write_bytes(_exposure_bytes)
_exposure_sys.path.insert(0, str(_exposure_code_root))
# Avoid importing an older helper cached by a previously executed release.
for _exposure_module in ("gan_exposure_data", "gan_exposure_training", "gan_exposure_runner", "gan_exposure_release"):
    _exposure_sys.modules.pop(_exposure_module, None)
from gan_exposure_runner import exposure_launch as _exposure_launch
_exposure_launch({"action":ACTION,"benchmark_c_updates":BENCHMARK_C_UPDATES,
    "max_workers":MAX_WORKERS,"gpu_ids":GPU_IDS,"session_hours":SESSION_HOURS,
    "save_every":SAVE_EVERY,"hsi_path":HSI_PATH,"gt_path":GT_PATH,
    "resume_run_path":RESUME_RUN_PATH,"output_root":OUTPUT_ROOT,"job_filter":JOB_FILTER},
    _exposure_code_root)
